# SOLARIS — ferramenta de estoque para lançamentos em Streamlit
### Google Colab + túnel `cloudflared` · mesmo passo a passo do notebook da disciplina

**Betina Moller e Julia Jallad · Orientador: Prof. Rodrigo Caiado · PUC-Rio - TCC**

Este notebook grava, **um arquivo por vez**, o aplicativo SOLARIS e o publica direto do Colab com um túnel temporário do
Cloudflare. Quando um produto é escolhido (da base) ou cadastrado (à mão ou por planilha), o app roda a simulação
de Monte Carlo, recomenda a política de estoque (P2, P3 ou P4) e monta o plano de ação de S&OE mês a mês. Cada rodada fica
salva no **Google Drive** e pode ser consultada e baixada depois.

| Tela | Pergunta que responde | Arquivo |
|---|---|---|
| 🏠 Início | O que é a ferramenta, como usar, quem fez, documentos | `tela_inicio.py` (textos em `conteudo.py`) |
| 📝 Cadastro | Quais produtos posso analisar? Como cadastrar um novo? | `tela_cadastro.py` |
| ✅ Recomendação | Qual política usar e por quê? Muda com a sensibilidade? | `tela_recomendacao.py` |
| 📅 Ação S&OE | O que fazer em cada mês e até quando decidir? | `tela_acao.py` |
| 📊 Análise avançada | Faixa de estoque, risco por mês, custo × serviço, backtest | `tela_analise.py` |
| 🗂️ Histórico | Rodadas salvas: reabrir, baixar, apagar | `tela_historico.py` |

**Arquivos**

```
Google Drive/MyDrive/SOLARIS/              ← pasta do projeto (dados e resultados ficam aqui)
    Triangulo Produtos V2.xlsx             ← base (você envia no passo 1)
    solaris_resultados.json                ← experimento anterior (segunda opinião) — opcional
    docs/                                  ← PDFs e documentos para baixar na tela de Início — opcional
    historico/                             ← cada rodada: .json (reabrir) + .xlsx (analisar) + _repeticoes.csv
    produtos_cadastrados.json              ← produtos novos cadastrados no app
/content (pasta do Colab)                  ← código do app, gravado pelas células %%writefile
    motor.py · conteudo.py · app.py · tela_*.py
```

### Como o app está organizado

1. **`motor.py` não tem nenhum comando `st.`**: lê a base, gera os futuros, simula as políticas, calcula as métricas,
   monta o plano de ação e salva o histórico. Dá para testar no próprio notebook (passo 4).
2. **`app.py` é o chassi**: barra lateral (produto, sensibilidade, custos, avançado), botão *Rodar simulação* e as 6 abas.
3. **Cada tela é um arquivo** com a mesma porta de entrada: `mostrar(dados, f)`.

## Passo 0 — Preparar o ambiente
O Colab já tem `pandas`, `numpy`, `plotly` e `openpyxl`. Instalamos o Streamlit e o Stockpyl (usado só na confirmação opcional).

In [ ]:
%pip install -q streamlit stockpyl  # Streamlit = app web; Stockpyl = simulador usado para confirmar o motor do SOLARIS

In [ ]:
import os, re, sys, time, shutil, subprocess, urllib.request  # utilitários para processos, arquivos e download
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

import streamlit
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| streamlit", streamlit.__version__)
try:
    import stockpyl
    print("stockpyl", stockpyl.__version__ if hasattr(stockpyl, "__version__") else "ok")
except ImportError:
    print("stockpyl não instalado: o app funciona, só a confirmação no Stockpyl fica desligada.")

Python 3.13.18 | pandas 2.2.3 | streamlit 1.67.0
stockpyl ok


## Passo 1 — Pasta do projeto no Google Drive

O histórico de rodadas e os produtos cadastrados ficam no Drive, para não sumirem quando o Colab desligar.
Na primeira vez, o Colab pede autorização para acessar o seu Drive.

- `USAR_DRIVE = False` usa uma pasta local do Colab (tudo se perde ao desconectar) — útil para um teste rápido.
- A célula seguinte abre a janela de upload **só se a base ainda não estiver na pasta**. Você pode enviar vários arquivos de
  uma vez: a base `.xlsx` e o `solaris_resultados.json` vão para a pasta do projeto; PDFs e `.docx` vão para `docs/`.

In [ ]:
USAR_DRIVE = True  # False = pasta local do Colab (não guarda o histórico entre sessões)

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA = Path("/content/drive/MyDrive/SOLARIS")
else:
    PASTA = Path("/content/SOLARIS")

for sub in ["", "docs", "historico"]:
    (PASTA / sub).mkdir(parents=True, exist_ok=True)
os.environ["SOLARIS_PASTA"] = str(PASTA)  # o motor.py e o app leem a pasta desta variável
print("Pasta do projeto:", PASTA)

Mounted at /content/drive
Pasta do projeto: /content/drive/MyDrive/SOLARIS


In [ ]:
BASE = PASTA / "Triangulo Produtos V2.xlsx"  # nome que o app espera encontrar


def enviar_arquivos():
    # Abre a janela de upload e guarda cada arquivo no lugar certo da pasta do projeto.
    from google.colab import files
    for nome, conteudo in files.upload().items():
        if nome.lower().endswith(".xlsx") and "triangulo" in nome.lower():
            destino = BASE                                   # garante o nome certo, mesmo que o navegador renomeie
        elif nome.lower().endswith(".json"):
            destino = PASTA / "solaris_resultados.json"
        elif nome.lower().endswith((".pdf", ".docx", ".pptx")):
            destino = PASTA / "docs" / nome
        else:
            destino = PASTA / nome
        destino.write_bytes(conteudo)
        print("salvo:", destino)


if not BASE.exists():
    print("Envie a base 'Triangulo Produtos V2.xlsx' (e, se quiser, o solaris_resultados.json e os PDFs).")
    enviar_arquivos()

# enviar_arquivos()  # descomente para enviar mais arquivos depois (ex.: novos PDFs para a tela de Início)

for f in sorted(PASTA.rglob("*")):
    if f.is_file() and "historico" not in f.parts:
        print(f"{f.relative_to(PASTA)}  ({f.stat().st_size / 1e3:,.0f} kB)")

Envie a base 'Triangulo Produtos V2.xlsx' (e, se quiser, o solaris_resultados.json e os PDFs).


Saving Triangulo Produtos V2.xlsx to Triangulo Produtos V2.xlsx
salvo: /content/drive/MyDrive/SOLARIS/Triangulo Produtos V2.xlsx
Triangulo Produtos V2.xlsx  (141 kB)
solaris_resultados.json  (163 kB)


**Conferir a base antes de desenhar telas.** A base tem a aba `Base` (atributos), a aba `BOM` (insumos) e uma aba por série
(`PRODUTO n` e `BENCH X_n`), cada uma com três triângulos: previsão (PV), plano de produção e estoque — linha = versão do
plano, coluna = mês-alvo.

In [ ]:
abas = pd.read_excel(BASE, sheet_name=None, header=None)
for nome, df in abas.items():
    print(f"{nome:14s} {df.shape[0]:>4} linhas × {df.shape[1]:>3} colunas")

Base              7 linhas ×   8 colunas
BOM              31 linhas ×   4 colunas
PRODUTO 1        55 linhas ×  21 colunas
BENCH A_1       115 linhas ×  41 colunas
BENCH B_1        85 linhas ×  31 colunas
PRODUTO 2        44 linhas ×  17 colunas
BENCH C_2        96 linhas ×  35 colunas
PRODUTO 3        55 linhas ×  21 colunas
BENCH D_3       115 linhas ×  41 colunas
BENCH E_3        85 linhas ×  31 colunas


## Passo 2 — Tema do app
Dourado no lugar do vermelho padrão do Streamlit.

In [ ]:
Path(".streamlit").mkdir(exist_ok=True)
Path(".streamlit/config.toml").write_text('[theme]\nprimaryColor = "#8C6D3F"\n\n[server]\nmaxUploadSize = 50\n')
print(Path(".streamlit/config.toml").read_text())

[theme]
primaryColor = "#8C6D3F"

[server]
maxUploadSize = 50



## Passo 3 — A lógica: `motor.py`

Toda conta fica aqui, **sem nenhum comando `st.`**. Principais blocos:

| Bloco | O que faz |
|---|---|
| 0. Constantes | valores-padrão (aparecem pré-preenchidos na barra lateral), z de 95/90/85%, regras dos gatilhos |
| 2. Base | lê os triângulos, infere o lote, monta o arquivo de erros dos benchmarks e a curva de incerteza por horizonte |
| 3. Produtos | produto da base (versão escolhida), produto novo, planilha modelo e leitura da planilha de cadastro |
| 4. Janela | aumentar = max(HB, LT insumo) + LT produção + mês de decisão; reduzir = HB + mês de decisão (meses, arredondando para cima) |
| 5. Cenários | demanda por *erros reais* (padrão), Normal ou Poisson; previsão revisada mês a mês; choque de fornecimento (Markov) |
| 6. Simulador | aplica P2/P3/P4/P2+lote e o plano programado em todos os futuros de uma vez (vetorizado), com pendência e retenção no choque |
| 7. Rodada | todos os cenários × políticas, resumo, regra de escolha (metas → menor custo; senão maior nível garantido) |
| 8. Ação S&OE | plano mês a mês, prazos para aumentar/reduzir, riscos e gatilhos (PV = 0 não aciona gatilho) |
| 9–12 | segunda opinião (séries parecidas), confirmação no Stockpyl, histórico no Drive, backtest com o histórico real |

In [ ]:
%%writefile motor.py
# motor.py — lógica do SOLARIS (sem nenhum comando de tela).
# Leitura da base, cadastro de produtos, simulação de Monte Carlo, políticas de estoque,
# métricas, regra de escolha, plano mês a mês, gatilhos de S&OE, confirmação no Stockpyl e histórico.
# Separar "conta" de "tela" deixa o app fácil de testar no notebook e garante que todas as telas usam as mesmas regras.
from __future__ import annotations

import io
import json
import math
import os
import re
import unicodedata
import zlib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

# =====================================================================================================
# 0. Constantes e valores-padrão (os valores-padrão aparecem pré-preenchidos na barra lateral)
# =====================================================================================================
PASTA = Path(os.environ.get("SOLARIS_PASTA", "."))          # pasta do projeto (no Colab: Google Drive)
ARQUIVO_BASE = PASTA / "Triangulo Produtos V2.xlsx"           # base
PASTA_HIST = PASTA / "historico"                               # rodadas salvas
ARQUIVO_PRODUTOS = PASTA / "produtos_cadastrados.json"        # produtos novos cadastrados
PASTA_DOCS = PASTA / "docs"                                    # PDFs para download na tela de início
ARQUIVOS_REFERENCIA = [PASTA / "solaris_resultados.json", PASTA / "resultados" / "solaris_resultados.json"]

DIAS_MES = 30
POLITICAS = ["P2", "P3", "P4", "P2L"]
NOME_POLITICA = {
    "P2": "P2 · Repor o previsto",
    "P3": "P3 · Previsto + reserva de segurança",
    "P4": "P4 · Reserva de segurança em lotes",
    "P2L": "P2 + lote · Repor o previsto em lotes",
    "PROG": "Plano programado (referência)",
    "P0": "P0 · Plano real (histórico)",
}
DESC_POLITICA = {
    "P2": "Todo mês produz a previsão até a produção chegar + a cobertura-alvo, descontando o estoque e o que já está a caminho. Sem reserva.",
    "P3": "Igual ao P2, mais uma reserva de segurança para o erro da previsão (z × σ).",
    "P4": "Igual ao P3, com a produção arredondada para múltiplos do lote mínimo.",
    "P2L": "Igual ao P2, com a produção arredondada para múltiplos do lote mínimo (sem reserva).",
}
DISTRIBUICOES = ["Erros reais (histórico)", "Normal", "Poisson"]
CHOQUES = ["Sem choque", "Com choque"]
PROTECAO_Z = {"95%": 1.645, "90%": 1.282, "85%": 1.036}

PADRAO = {
    "n_rep": 500, "protecao": "95%", "politicas": ["P2", "P3", "P4"],
    "distribuicoes": ["Erros reais (histórico)"], "choques": ["Sem choque"],
    # custos (visíveis na barra lateral)
    "custo_estoque_aa": 0.25, "custo_falta": 1.00, "custo_sobra": 0.30, "custo_excesso": 0.05,
    # avançado
    "regra_janela": "Assimétrica (validada)", "mes_decisao": True, "lt_producao": 1,
    "mult_incerteza": 1.0, "choque_inicio": 0.10, "choque_fim": 0.50,
    "meta_servico": 0.95, "meta_garantido": 0.90, "cob_min": 30, "cob_max": 180,
    "semente": 2026, "stockpyl": False, "n_rep_stockpyl": 50, "hbs": None,
}
# Gatilhos da Ação S&OE (edite aqui se a regra mudar)
GATILHO = {"risco_critico": 0.50, "risco_atencao": 0.25, "risco_excesso_atencao": 0.50, "desvio_atencao": 0.15,
           "tolerancia_manter": 0.05}

# Cores do semáforo e das políticas (mesmas em todas as telas)
CORES = {"Crítico": "#C0392B", "Atenção": "#D68910", "OK": "#1E8449", "—": "#7F8C8D"}
COR_POLITICA = {"P2": "#2a78d6", "P3": "#eb6834", "P4": "#1baf7a", "P2L": "#eda100", "PROG": "#8a8f98", "P0": "#8a8f98"}

_MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
_MESES_PT = {m: i + 1 for i, m in enumerate(_MESES)}
_MES_RE = re.compile(r"^([a-zç]{3})/(\d{4})$")


# =====================================================================================================
# 1. Utilidades de formatação
# =====================================================================================================
def pct(v, casas=1):
    return "—" if v is None or (isinstance(v, float) and not np.isfinite(v)) else f"{100 * v:.{casas}f}%".replace(".", ",")


def num(v, casas=0):
    if v is None or (isinstance(v, float) and not np.isfinite(v)):
        return "—"
    return f"{v:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def mi(v, casas=2):
    return "—" if v is None or not np.isfinite(v) else "R$ " + num(v / 1e6, casas) + " mi"


def norm_txt(s):
    s = unicodedata.normalize("NFD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", s.lower())


def mes_label(v):
    """Converte 'ago/2025', datas do Excel, '2025-08' ou '08/2025' em 'ago/2025'."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, (pd.Timestamp, datetime)):
        return f"{_MESES[v.month - 1]}/{v.year}"
    s = str(v).strip().lower()
    m = _MES_RE.match(s)
    if m and m.group(1) in _MESES_PT:
        return s
    m = re.match(r"^(\d{4})[-/](\d{1,2})", s)
    if m:
        return f"{_MESES[int(m.group(2)) - 1]}/{m.group(1)}"
    m = re.match(r"^(\d{1,2})[-/](\d{4})$", s)
    if m:
        return f"{_MESES[int(m.group(1)) - 1]}/{m.group(2)}"
    return None


def seq_meses(inicio, n):
    m = _MES_RE.match(mes_label(inicio) or "jan/2026")
    i0, ano = _MESES_PT[m.group(1)] - 1, int(m.group(2))
    return [f"{_MESES[(i0 + k) % 12]}/{ano + (i0 + k) // 12}" for k in range(n)]


def meses_para(semanas_ou_dias, unidade="dias"):
    """HB e LT em meses, sempre arredondando para cima (16 semanas → 4 meses; 60 dias → 2 meses)."""
    dias = float(semanas_ou_dias) * (7 if unidade == "semanas" else 1)
    return int(math.ceil(dias / DIAS_MES - 1e-9)) if dias > 0 else 0


# =====================================================================================================
# 2. Leitura da base (formato em triângulo) e curva de incerteza
# =====================================================================================================
def _eh_mes(x):
    return isinstance(x, str) and _MES_RE.match(x.strip().lower()) is not None


def _periodo(label):
    m = _MES_RE.match(label.strip().lower())
    return pd.Period(year=int(m.group(2)), month=_MESES_PT[m.group(1)], freq="M")


def _ler_aba_base(caminho):
    raw = pd.read_excel(caminho, sheet_name="Base", header=None)
    lin = next(i for i, r in raw.iterrows() if any(str(v).strip().upper() == "PRODUTO" for v in r.values))
    cab = [str(v).strip() if pd.notna(v) else "" for v in raw.iloc[lin].values]
    df = raw.iloc[lin + 1:].copy()
    df.columns = cab
    df = df.loc[:, [c for c in df.columns if c]].dropna(subset=["PRODUTO"])
    df = df.rename(columns={"PRODUTO": "product_id", "Origem Produção": "origem", "Cobertura Target (dias)": "cobertura_dias",
                            "HB (dias)": "hb_dias", "Maior LT de Insumo (in weeks)": "lt_insumo_semanas", "R$ PF": "preco",
                            "BENCHS": "benchmarks"})
    df["product_id"] = df["product_id"].apply(lambda v: str(int(v)) if isinstance(v, (int, float)) else str(v).strip())
    for c in ["cobertura_dias", "hb_dias", "lt_insumo_semanas", "preco"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["origem"] = df["origem"].astype(str).str.strip()
    return df.reset_index(drop=True)


def _ler_triangulo(caminho, aba):
    """Lê os três blocos (PV, plano de produção, estoque). Linha = versão do plano; coluna = mês-alvo."""
    raw = pd.read_excel(caminho, sheet_name=aba, header=None)
    blocos, cab, atual = {}, {}, None
    for _, r in raw.iterrows():
        a = r.iloc[0]
        if not isinstance(a, str):
            continue
        s = a.strip(); su = s.upper()
        if su.startswith("PV"):
            atual = "previsao"; blocos[atual] = []
        elif su.startswith("PLANO"):
            atual = "producao"; blocos[atual] = []
        elif su.startswith("ESTOQUE"):
            atual = "estoque"; blocos[atual] = []
        elif s == "Mes/Ano" and atual and _eh_mes(r.iloc[1]):
            cab[atual] = [str(x).strip().lower() for x in r.iloc[1:] if _eh_mes(x)]
        elif _eh_mes(s) and atual in cab:
            n = len(cab[atual])
            blocos[atual].append((s.lower(), pd.to_numeric(r.iloc[1:1 + n], errors="coerce").to_numpy(dtype=float)))
    return {k: pd.DataFrame([v for _, v in blocos[k]], index=[i for i, _ in blocos[k]], columns=cab[k])
            for k in ("previsao", "producao", "estoque")}


def inferir_lote(producao: pd.DataFrame, tol=0.02, fracao_min=0.80):
    """Lote = maior q tal que ≥ 80% das quantidades PLANEJADAS são múltiplos de q (tolerância 2%; mínimo 5 quantidades)."""
    cols, plan = list(producao.columns), []
    for v in producao.index:
        if v in cols:
            plan += [x for x in producao.loc[v].iloc[cols.index(v):].to_numpy() if np.isfinite(x) and x > 0]
    plan = np.array(plan, dtype=float)
    if len(plan) < 5:
        return 0
    melhor = 0
    for c in np.unique(np.round(plan))[:10]:
        r = plan / c
        if np.mean(np.abs(r - np.round(r)) <= tol * np.maximum(1, np.round(r))) >= fracao_min:
            melhor = max(melhor, int(round(c)))
    return melhor


def _matrizes(tri):
    """Matrizes alinhadas no tempo; células realizadas (à esquerda da diagonal) ficam vazias (sem vazamento)."""
    F, PP, INV = tri["previsao"], tri["producao"], tri["estoque"]
    alvos, versoes = list(F.columns), list(F.index)
    off = [_periodo(c) for c in alvos].index(_periodo(versoes[0]))
    alvos = alvos[off:]
    T = alvos.index(versoes[-1])
    nC = len(alvos)

    def mat(df):
        M = np.full((len(versoes), nC), np.nan)
        for r, v in enumerate(versoes):
            M[r, r:] = df.loc[v, alvos].to_numpy(dtype=float)[r:]
        return M

    return {"meses": alvos, "versoes": versoes, "T_real": T, "V": mat(F), "PP": mat(PP),
            "realizado": F.loc[versoes[-1], alvos[:T]].to_numpy(dtype=float),
            "producao_real": PP.loc[versoes[-1], alvos[:T]].to_numpy(dtype=float),
            "estoque_real": INV.loc[versoes[-1], alvos[:T]].to_numpy(dtype=float)}


def carregar_base(caminho=ARQUIVO_BASE):
    """Lê a base inteira: atributos, BOM e as abas PRODUTO n / BENCH X_n. Devolve séries, arquivo de erros e curva."""
    caminho = str(caminho)
    base = _ler_aba_base(caminho)
    bom = pd.read_excel(caminho, sheet_name="BOM")
    bom.columns = ["product_id", "material_id", "material_desc", "lt_semanas"]
    bom["product_id"] = bom["product_id"].apply(lambda v: str(int(v)) if isinstance(v, (int, float)) else str(v).strip())
    series = {}
    for aba in pd.ExcelFile(caminho).sheet_names:
        s = aba.strip().upper()
        mp, mb = re.match(r"^PRODUTO\s*(\d+)$", s), re.match(r"^BENCH\s*([A-Z]+)\s*_\s*(\d+)$", s)
        if not (mp or mb):
            continue
        sku, tipo, pid = (f"PRODUTO {mp.group(1)}", "produto", mp.group(1)) if mp else (f"BENCH {mb.group(1)}", "benchmark", mb.group(2))
        b = base.loc[base["product_id"] == pid].iloc[0]
        tri = _ler_triangulo(caminho, aba)
        m = _matrizes(tri)
        lt_bom = bom.loc[bom["product_id"] == pid, "lt_semanas"].max()
        series[sku] = {"sku": sku, "tipo": tipo, "product_id": pid, "origem": b["origem"],
                       "cobertura_dias": float(b["cobertura_dias"]), "hb_dias": float(b["hb_dias"]),
                       "lt_insumo_semanas": float(np.nanmax([b["lt_insumo_semanas"], lt_bom])),
                       "preco": float(b["preco"]), "lote": inferir_lote(tri["producao"]), **m}
    arquivo = arquivo_erros(series)
    return {"series": series, "arquivo": arquivo, "curva": curva_incerteza(arquivo)}


def arquivo_erros(series, excluir=None, max_h=12):
    """Erros de previsão dos benchmarks (lançamentos com realizado conhecido): um registro por versão × mês-alvo."""
    linhas = []
    for sku, s in series.items():
        if s["tipo"] != "benchmark" or sku == excluir:
            continue
        V, A = s["V"], s["realizado"]
        for r in range(min(V.shape[0], len(A))):
            for c in range(r, len(A)):
                f, a = V[r, c], A[c]
                if not np.isfinite(f) or (f == 0 and a == 0) or c - r > max_h:
                    continue
                linhas.append({"sku": sku, "versao": r, "alvo": c, "h": c - r, "previsao": f, "realizado": a,
                               "erro": a - f, "erro_rel": (a - f) / f if f > 0 else np.nan})
    return pd.DataFrame(linhas)


def curva_incerteza(arq):
    """CV do erro por horizonte = desvio-padrão do erro ÷ previsão média (interpolado, limitado a [0,01; 1])."""
    g = arq.groupby("h").agg(prev_media=("previsao", "mean"), erro_dp=("erro", "std"), n=("erro", "size"))
    g["cv"] = (g["erro_dp"] / g["prev_media"].replace(0, np.nan)).interpolate().bfill().ffill().clip(0.01, 1.0)
    return g.reset_index()[["h", "cv", "n"]]


def cv_por_horizonte(curva, n):
    tab = dict(zip(curva["h"].astype(int), curva["cv"]))
    ult = float(curva.sort_values("h")["cv"].iloc[-1])
    return np.array([tab.get(h, ult) for h in range(n + 1)])


def pools_erros(arq):
    """Erros relativos reais: por horizonte (sorteio independente) e trajetórias inteiras (mantém a persistência do erro)."""
    a = arq.dropna(subset=["erro_rel"])
    por_h = {int(h): g["erro_rel"].clip(-0.95, 3.0).to_numpy() for h, g in a.groupby("h")}
    traj = []
    for (_, _), g in a.groupby(["sku", "versao"]):
        v = np.full(13, np.nan)
        for h, e in zip(g["h"], g["erro_rel"].clip(-0.95, 3.0)):
            if h <= 12:
                v[int(h)] = e
        if np.isfinite(v).sum() >= 3:
            traj.append(v)
    return por_h, traj


# =====================================================================================================
# 3. Produtos: da base (escolhendo a versão da previsão) ou cadastrados pelo usuário
# =====================================================================================================
CAMPOS = ["nome", "origem", "cobertura_dias", "hb_dias", "lt_insumo_semanas", "preco", "lote", "estoque_inicial"]


def produto_da_base(serie, versao_idx=0):
    """Monta o produto a partir de uma versão do triângulo: previsão e plano dessa versão, estoque real anterior."""
    r = int(versao_idx)
    V, PP = serie["V"], serie["PP"]
    prev, prog = np.nan_to_num(V[r, r:]), np.nan_to_num(PP[r, r:])
    meses = serie["meses"][r:]
    k = len(prev)
    while k > 1 and prev[k - 1] == 0 and prog[k - 1] == 0:
        k -= 1
    est = float(serie["estoque_real"][r - 1]) if 0 < r <= len(serie["estoque_real"]) else 0.0
    return {"id": f"base::{serie['sku']}::{r}", "nome": f"{serie['sku']} (versão {serie['versoes'][r]})", "fonte": "base",
            "sku_base": serie["sku"], "versao": r, "origem": serie["origem"], "cobertura_dias": serie["cobertura_dias"],
            "hb_dias": serie["hb_dias"], "lt_insumo_semanas": serie["lt_insumo_semanas"], "preco": serie["preco"],
            "lote": int(serie["lote"]), "estoque_inicial": max(est, 0.0), "meses": meses[:k],
            "previsao": [float(x) for x in prev[:k]], "programada": [float(x) for x in prog[:k]]}


def versoes_validas(serie):
    """Versões com pelo menos 2 meses de previsão positiva (antes disso o lançamento ainda não tinha previsão)."""
    return [i for i in range(len(serie["versoes"])) if np.sum(np.nan_to_num(serie["V"][i, i:]) > 0) >= 2]


def produto_novo(nome="Novo produto", inicio="jan/2027", n=12):
    return {"id": f"novo::{nome}", "nome": nome, "fonte": "novo", "origem": "Interno", "cobertura_dias": 60, "hb_dias": 30,
            "lt_insumo_semanas": 8, "preco": 20.0, "lote": 0, "estoque_inicial": 0.0, "meses": seq_meses(inicio, n),
            "previsao": [0.0] * n, "programada": [0.0] * n}


def validar_produto(p):
    erros = []
    if not str(p.get("nome", "")).strip():
        erros.append("Informe o nome do produto.")
    if sum(1 for x in p["previsao"] if x > 0) < 2:
        erros.append("Informe pelo menos 2 meses com previsão de vendas positiva.")
    if float(p["preco"]) <= 0:
        erros.append("Informe o preço (R$ PF) maior que zero.")
    if len(p["meses"]) != len(p["previsao"]) or len(p["previsao"]) != len(p["programada"]):
        erros.append("Meses, previsão e produção programada precisam ter o mesmo tamanho.")
    return erros


def carregar_cadastrados():
    try:
        return json.loads(ARQUIVO_PRODUTOS.read_text(encoding="utf-8"))
    except Exception:
        return []


def salvar_cadastrados(lista):
    ARQUIVO_PRODUTOS.parent.mkdir(parents=True, exist_ok=True)
    ARQUIVO_PRODUTOS.write_text(json.dumps(lista, ensure_ascii=False, indent=1), encoding="utf-8")


COLS_ATR = ["Produto", "Origem (Interno/Terceiro)", "Cobertura-alvo (dias)", "HB (dias)", "Maior LT de insumo (semanas)",
            "R$ PF (preço por unidade)", "Lote (un.; 0 = sem lote)", "Estoque atual (un.)"]
COLS_PREV = ["Produto", "Mês (mmm/aaaa)", "Previsão (un.)", "Produção programada (un.)"]


def planilha_modelo(produtos=None):
    """Planilha modelo de importação (abas Instrucoes, Atributos, Previsoes), em bytes para download."""
    produtos = produtos or []
    instr = pd.DataFrame({"Como preencher": [
        "1. Aba Atributos: uma linha por produto. Não mude os títulos das colunas.",
        "2. Aba Previsoes: uma linha por produto e por mês: previsão de vendas e produção programada para chegar no mês.",
        "3. O nome do produto deve ser igual nas duas abas. Meses no formato mmm/aaaa (ex.: ago/2025) ou data do Excel.",
        "4. Comece pelo mês atual e informe meses suficientes para cobrir a janela de reação + a cobertura-alvo.",
        "5. Origem: Interno ou Terceiro. HB em dias (30 ou 60). LT de insumo em semanas.",
        "6. Na ferramenta: Cadastro → Subir planilha. Produtos com o mesmo nome são atualizados."]})
    atr = pd.DataFrame([[p["nome"], p["origem"], p["cobertura_dias"], p["hb_dias"], p["lt_insumo_semanas"], p["preco"],
                         p["lote"], p["estoque_inicial"]] for p in produtos], columns=COLS_ATR)
    prev = pd.DataFrame([[p["nome"], m, f, g] for p in produtos for m, f, g in zip(p["meses"], p["previsao"], p["programada"])],
                        columns=COLS_PREV)
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as w:
        instr.to_excel(w, sheet_name="Instrucoes", index=False)
        atr.to_excel(w, sheet_name="Atributos", index=False)
        prev.to_excel(w, sheet_name="Previsoes", index=False)
        for ws in w.book.worksheets:
            for col in ws.columns:
                ws.column_dimensions[col[0].column_letter].width = 26 if ws.title != "Instrucoes" else 110
    return buf.getvalue()


def ler_planilha_produtos(arquivo):
    """Lê a planilha modelo (ou uma parecida). Devolve (lista de produtos, avisos)."""
    abas = pd.read_excel(arquivo, sheet_name=None)
    nome_aba = lambda chaves: next((n for n in abas if any(k in norm_txt(n) for k in chaves)), None)
    a_atr, a_prev = nome_aba(["atribut", "attr"]), nome_aba(["previs", "forecast", "demanda"])
    if not a_atr or not a_prev:
        return [], ["Não encontrei as abas 'Atributos' e 'Previsoes'. Use a planilha modelo."]
    A, P = abas[a_atr], abas[a_prev]
    col = lambda df, testes: next((c for c in df.columns if any(t(norm_txt(c)) for t in testes)), None)
    ca = {"nome": col(A, [lambda h: h.startswith("produto"), lambda h: h in ("nome", "sku")]),
          "origem": col(A, [lambda h: h.startswith("origem")]), "cob": col(A, [lambda h: "cobertura" in h]),
          "hb": col(A, [lambda h: h.startswith("hb")]), "lt": col(A, [lambda h: "insumo" in h or h.startswith("lt")]),
          "preco": col(A, [lambda h: "pf" in h or "preco" in h]), "lote": col(A, [lambda h: h.startswith("lote")]),
          "est": col(A, [lambda h: "estoque" in h])}
    cp = {"nome": col(P, [lambda h: h.startswith("produto"), lambda h: h in ("nome", "sku")]),
          "mes": col(P, [lambda h: h.startswith("mes") or h.startswith("data")]),
          "prev": col(P, [lambda h: h.startswith("previs") or "demanda" in h]),
          "prog": col(P, [lambda h: "program" in h or h.startswith("producao")])}
    if not ca["nome"] or not cp["nome"] or not cp["mes"] or not cp["prev"]:
        return [], ["Colunas obrigatórias não encontradas (Produto, Mês, Previsão)."]
    avisos, produtos = [], []
    numf = lambda v, d=0.0: float(pd.to_numeric(v, errors="coerce")) if pd.notna(pd.to_numeric(v, errors="coerce")) else d
    for _, r in A.iterrows():
        nome = str(r[ca["nome"]]).strip()
        if not nome or nome.lower() == "nan":
            continue
        sub = P[P[cp["nome"]].astype(str).str.strip() == nome].copy()
        sub["_m"] = sub[cp["mes"]].apply(mes_label)
        if sub["_m"].isna().any():
            avisos.append(f"{nome}: há meses em formato inválido (ignorados).")
        sub = sub.dropna(subset=["_m"])
        sub["_p"] = sub["_m"].apply(_periodo)
        sub = sub.sort_values("_p")
        if sub.empty:
            avisos.append(f"{nome}: sem linhas na aba Previsoes.")
            continue
        orig = norm_txt(r[ca["origem"]]) if ca["origem"] else "interno"
        produtos.append({"id": f"novo::{nome}", "nome": nome, "fonte": "novo",
                         "origem": "Terceiro" if orig.startswith("terc") else "Interno",
                         "cobertura_dias": numf(r[ca["cob"]], 60) if ca["cob"] else 60,
                         "hb_dias": numf(r[ca["hb"]], 30) if ca["hb"] else 30,
                         "lt_insumo_semanas": numf(r[ca["lt"]], 0) if ca["lt"] else 0,
                         "preco": numf(r[ca["preco"]], 0) if ca["preco"] else 0,
                         "lote": int(numf(r[ca["lote"]], 0)) if ca["lote"] else 0,
                         "estoque_inicial": numf(r[ca["est"]], 0) if ca["est"] else 0,
                         "meses": list(sub["_m"]), "previsao": [numf(x) for x in sub[cp["prev"]]],
                         "programada": [numf(x) for x in sub[cp["prog"]]] if cp["prog"] else [0.0] * len(sub)})
    return produtos, avisos


# =====================================================================================================
# 4. Janela de reação
# =====================================================================================================
def janelas(prod, par):
    """Devolve (janela para AUMENTAR, janela para REDUZIR), em meses.
    Assimétrica (validada): aumentar = max(HB, LT de insumo) + LT de produção + mês de decisão;
                            reduzir  = HB + mês de decisão (não depende de insumo nem de produção).
    Simples (experimento anterior): max(HB, LT de produção) nos dois sentidos."""
    hb = meses_para(prod["hb_dias"], "dias")
    lt = meses_para(prod["lt_insumo_semanas"], "semanas")
    d = 1 if par["mes_decisao"] else 0
    if par["regra_janela"].startswith("Assim"):
        up, down = max(hb, lt) + int(par["lt_producao"]) + d, hb + d
    else:
        up = down = max(hb, int(par["lt_producao"]), 1)
    down = max(1, down)
    return max(up, down), down


# =====================================================================================================
# 5. Cenários de Monte Carlo: demanda, previsão revisada mês a mês e choque de fornecimento
# =====================================================================================================
def gerar_cenarios(prev, dist, R, rng, cv, por_h, traj, mult, choque, p_ini, p_fim):
    """Gera R futuros possíveis. Cada futuro tem:
       D[r, j]     demanda que acontece no mês j;
       Vr[r, t, j] previsão que o planejador tem no mês t para o mês j (revisada a cada mês; Vr[:,0,:] = previsão cadastrada);
       S[r, j]     True se o mês j está em choque (recebimento retido)."""
    F = np.asarray(prev, float)
    n = len(F)
    hmax = lambda h: min(int(h), 12)
    if dist.startswith("Erros"):
        E = np.empty((R, n))
        for r in range(R):
            tr = traj[rng.integers(len(traj))] if traj else np.full(13, np.nan)
            for j in range(n):
                e = tr[hmax(j)]
                if not np.isfinite(e):
                    pool = por_h.get(hmax(j)) if por_h.get(hmax(j)) is not None else np.concatenate(list(por_h.values()))
                    e = pool[rng.integers(len(pool))]
                E[r, j] = e
        D = np.clip(F * (1 + mult * E), 0, None)
    elif dist == "Normal":
        D = np.clip(F * (1 + rng.normal(0, 1, (R, n)) * mult * cv[:n]), 0, None)
    else:  # Poisson
        D = rng.poisson(np.clip(F, 0, None), (R, n)).astype(float)
    Vr = np.zeros((R, n, n))
    Vr[:, 0, :] = F
    for t in range(1, n):
        for j in range(t + 1, n):
            h = j - t
            if dist.startswith("Erros"):
                pool = por_h.get(hmax(h)) if por_h.get(hmax(h)) is not None else np.concatenate(list(por_h.values()))
                e = pool[rng.integers(len(pool), size=R)] * mult
                Vr[:, t, j] = D[:, j] / np.clip(1 + e, 0.2, None)
            elif dist == "Normal":
                Vr[:, t, j] = np.clip(D[:, j] * (1 + rng.normal(0, 1, R) * mult * cv[h]), 0, None)
            else:
                Vr[:, t, j] = np.clip(D[:, j] + rng.normal(0, 1, R) * np.sqrt(np.maximum(D[:, j], 1)), 0, None)
        Vr[:, t, F == 0] = 0.0          # meses sem previsão continuam sem previsão
    S = np.zeros((R, n), bool)
    if choque == "Com choque":
        estado = np.zeros(R, bool)
        for j in range(n):
            u = rng.random(R)
            estado = np.where(estado, u >= p_fim, u < p_ini)
            S[:, j] = estado
    return D, Vr, S


# =====================================================================================================
# 6. Simulador do SOLARIS (vetorizado nas R repetições) — política, falta como pendência, métricas
# =====================================================================================================
def _cobertura(estoque, Vt, t, n, cob_max):
    """Cobertura para frente (dias): quantos dias o estoque dura consumindo a previsão dos meses seguintes.
    Se a previsão acabar antes do estoque: > cob_max quando o horizonte restante já passa de cob_max; senão indefinida."""
    R = len(estoque)
    cov = np.full(R, np.nan)
    resto = estoque.copy()
    dias = np.zeros(R)
    aberto = np.ones(R, bool)
    for k in range(t + 1, n):
        f = Vt[:, k]
        consome = np.minimum(resto, f)
        acaba = aberto & (resto < f) & (f > 0)
        cov[acaba] = dias[acaba] + DIAS_MES * (resto[acaba] / f[acaba])
        aberto &= ~acaba
        resto = resto - consome
        dias = dias + DIAS_MES
    sobra = aberto & (dias >= cob_max)
    cov[sobra] = dias[sobra] + 1          # dura mais que o horizonte e que o limite de excesso
    return cov


def _excesso_unidades(estoque, Vt, t, n, cob_max):
    k = int(round(cob_max / DIAS_MES))
    if t + k >= n:
        return np.zeros(len(estoque))
    return np.maximum(0.0, estoque - Vt[:, t + 1:t + 1 + k].sum(axis=1))


def simular(prod, pol, D, Vr, S, par, cv, z):
    """Executa a política em todos os futuros de uma vez. Devolve métricas por repetição e por mês."""
    R, n = D.shape
    F = np.asarray(prod["previsao"], float)
    prog = np.asarray(prod["programada"], float)
    up, down = janelas(prod, par)
    lote = int(prod.get("lote") or 0)
    usa_lote = pol in ("P4", "P2L") and lote > 0
    zz = z if pol in ("P3", "P4") else 0.0
    c = float(prod["cobertura_dias"]) / DIAS_MES
    inteiro, frac = int(math.floor(c)), c - math.floor(c)
    n_prot = max(up + int(math.ceil(c)), up + 1)
    preco = float(prod["preco"])
    h_un, b_un = par["custo_estoque_aa"] / 12 * preco, par["custo_falta"] * preco
    L = n + up + 2
    arr = np.zeros((R, L))
    kp = int(np.nonzero(prog > 0)[0].max()) if (prog > 0).any() else -1      # último mês com produção programada
    if pol == "PROG":
        # Plano programado: segue exatamente o programado até o último mês com plano; depois disso continua repondo
        # o previsto (P2), porque o plano ainda não foi feito para esses meses.
        arr[:, :kp + 1] = prog[:kp + 1]
    else:
        arr[:, :min(up, n)] = prog[:min(up, n)]          # o que já está programado dentro da janela de aumento
    IL = np.full(R, float(prod["estoque_inicial"]))
    retido = np.zeros(R)
    out = {k: np.zeros((R, n)) for k in ["atendido", "estoque", "pendente", "cob", "ss", "excesso", "pedido", "corte", "recebido"]}
    cvw = cv[1:n_prot + 1] * par["mult_incerteza"] if len(cv) > n_prot else np.resize(cv[1:], n_prot) * par["mult_incerteza"]
    for t in range(n):
        # 1) recebimento (com choque, o que chega fica retido até o choque terminar)
        chega = arr[:, t]
        retido = retido + np.where(S[:, t], chega, 0.0)
        receb = np.where(S[:, t], 0.0, chega + retido)
        retido = np.where(S[:, t], retido, 0.0)
        # 2) demanda: atende com o estoque disponível; o resto vira pendência
        disp = IL + receb
        atend = np.minimum(D[:, t], np.maximum(disp, 0.0))
        IL = disp - D[:, t]
        Vt = Vr[:, t, :] if t > 0 else np.broadcast_to(F, (R, n))
        ss = np.zeros(R)
        if pol != "PROG" or t + up > kp:
            # 3) nível-alvo com a previsão vigente: até a produção chegar + cobertura-alvo (+ reserva)
            def janela(a, k):
                b = min(n, a + k)
                return Vt[:, a:b].sum(axis=1) if b > a else np.zeros(R)
            pre = janela(t + 1, up)
            cob = janela(t + 1 + up, inteiro)
            j_f = t + 1 + up + inteiro
            if frac > 1e-9 and j_f < n:
                cob = cob + frac * Vt[:, j_f]
            if zz > 0:
                b = min(n, t + 1 + n_prot)
                fut = Vt[:, t + 1:b]
                sig = np.sqrt(fut.sum(axis=1)) if par["_dist"] == "Poisson" else np.sqrt(((fut * cvw[:fut.shape[1]]) ** 2).sum(axis=1))
                ss = zz * sig
            alvo = pre + cob + ss
            pos = IL + retido + arr[:, t + 1:].sum(axis=1)          # posição de estoque
            q = np.maximum(0.0, alvo - pos)
            if usa_lote:
                q = np.ceil(q / lote - 1e-9) * lote
            if t + up < L:
                arr[:, t + up] += q
            out["pedido"][:, t] = q
        if pol != "PROG":
            # 4) redução: o que está planejado fora da janela de redução pode ser cortado (em lotes inteiros, se houver lote)
            exc = np.maximum(0.0, pos - alvo)
            cortado = np.zeros(R)
            for m in range(min(t + up - 1, L - 1), t + down - 1, -1):
                corte = np.minimum(arr[:, m], exc)
                if usa_lote:
                    corte = np.floor(corte / lote + 1e-9) * lote
                arr[:, m] -= corte
                exc -= corte
                cortado += corte
            out["corte"][:, t] = cortado
        est = np.maximum(IL, 0.0)
        out["atendido"][:, t] = atend
        out["estoque"][:, t] = est
        out["pendente"][:, t] = np.maximum(-IL, 0.0)
        out["ss"][:, t] = ss
        out["recebido"][:, t] = receb
        out["cob"][:, t] = _cobertura(est, Vt, t, n, par["cob_max"])
        out["excesso"][:, t] = _excesso_unidades(est, Vt, t, n, par["cob_max"])
    out["chegada_final"] = arr[:, :n].copy()
    return metricas(out, D, F, prod, par, h_un, b_un, preco)


def metricas(out, D, F, prod, par, h_un, b_un, preco):
    """Métricas por repetição. Meses sem previsão de venda ficam fora das métricas de avaliação."""
    aval = np.asarray(par["_aval"], bool) if par.get("_aval") is not None else F > 0
    Da, Aa = D[:, aval], out["atendido"][:, aval]
    tot = Da.sum(axis=1)
    falta_mes = (Aa < Da - 1e-6) & (Da > 0)
    cov = out["cob"][:, aval]
    defin = np.isfinite(cov)
    with np.errstate(invalid="ignore", divide="ignore"):
        sub = np.where(defin.sum(axis=1) > 0, ((cov < par["cob_min"]) & defin).sum(axis=1) / np.maximum(defin.sum(axis=1), 1), 0.0)
        sobre = np.where(defin.sum(axis=1) > 0, ((cov > par["cob_max"]) & defin).sum(axis=1) / np.maximum(defin.sum(axis=1), 1), 0.0)
    custo_est = (out["estoque"] * h_un).sum(axis=1)
    custo_falta = (out["pendente"] * b_un).sum(axis=1)
    custo_sobra = out["estoque"][:, -1] * par["custo_sobra"] * preco
    custo_exc = (out["excesso"] * par["custo_excesso"] * preco).sum(axis=1)
    rep = pd.DataFrame({
        "nivel_servico": np.where(tot > 0, Aa.sum(axis=1) / np.maximum(tot, 1e-9), 1.0),
        "meses_falta": falta_mes.mean(axis=1) if aval.any() else 0.0,
        "meses_cob_baixa": sub, "meses_excesso": sobre,
        "estoque_medio": out["estoque"][:, aval].mean(axis=1) if aval.any() else 0.0,
        "estoque_final": out["estoque"][:, -1], "reserva_media": out["ss"][:, aval].mean(axis=1) if aval.any() else 0.0,
        "custo_estoque": custo_est, "custo_falta": custo_falta, "custo_sobra": custo_sobra, "custo_excesso": custo_exc,
        "custo_total": custo_est + custo_falta + custo_sobra + custo_exc,
        "pedidos": (out["pedido"] > 0).sum(axis=1), "unidades_cortadas": out["corte"].sum(axis=1)})
    mensal = pd.DataFrame({
        "estoque_p5": np.percentile(out["estoque"], 5, axis=0), "estoque_p50": np.percentile(out["estoque"], 50, axis=0),
        "estoque_p95": np.percentile(out["estoque"], 95, axis=0),
        "prob_falta": ((out["atendido"] < D - 1e-6) & (D > 0)).mean(axis=0),
        # cobertura indefinida (a previsão acaba antes do estoque e o horizonte restante é curto) em mais da metade dos
        # futuros → probabilidade não calculada (NaN), para não mostrar "0% de excesso" no fim do horizonte
        "prob_excesso": np.where(np.isfinite(out["cob"]).mean(axis=0) >= 0.5,
                                 (np.nan_to_num(out["cob"], nan=0) > par["cob_max"]).sum(axis=0) / np.maximum(np.isfinite(out["cob"]).sum(axis=0), 1), np.nan),
        "prob_cob_baixa": np.where(np.isfinite(out["cob"]).mean(axis=0) >= 0.5,
                                   ((out["cob"] < par["cob_min"]) & np.isfinite(out["cob"])).sum(axis=0) / np.maximum(np.isfinite(out["cob"]).sum(axis=0), 1), np.nan),
        "chegada_media": out["chegada_final"].mean(axis=0), "demanda_media": D.mean(axis=0)})
    return {"rep": rep, "mensal": mensal, "_bruto": out}


# =====================================================================================================
# 7. Rodada completa: cenários de sensibilidade (distribuição × choque × HB) × políticas × repetições
# =====================================================================================================
def _semente(base, *partes):
    return int(base) + zlib.crc32("|".join(map(str, partes)).encode()) % 1_000_000 * 10


def nome_cenario(dist, choque, hb):
    return f"{dist} · {choque} · HB {int(hb)} dias"


def rodar(prod, par, base_dados, progresso=None):
    """Roda o Monte Carlo para o produto em todos os cenários escolhidos. Devolve um dicionário com tudo."""
    n = len(prod["previsao"])
    excluir = prod.get("sku_base") if prod.get("fonte") == "base" else None
    # deixa-um-fora: um benchmark não calibra o próprio erro
    arq = arquivo_erros(base_dados["series"], excluir=excluir) if excluir and excluir.startswith("BENCH") else base_dados["arquivo"]
    curva = curva_incerteza(arq)
    cv = cv_por_horizonte(curva, n + 40)
    por_h, traj = pools_erros(arq)
    z = PROTECAO_Z[par["protecao"]]
    pols = [p for p in par["politicas"] if p != "P2L" or int(prod.get("lote") or 0) > 0]
    tem_prog = sum(prod["programada"]) > 0
    hbs = par.get("hbs") or [prod["hb_dias"]]
    cen_def = {}
    for d in par["distribuicoes"]:
        for c in par["choques"]:
            for hb in hbs:
                p_hb = dict(prod, hb_dias=float(hb))
                up, down = janelas(p_hb, par)
                cen_def[nome_cenario(d, c, hb)] = {"distribuicao": d, "choque": c, "hb_dias": float(hb),
                                                    "janela_aumento": up, "janela_reducao": down}
    resumo, reps, mensal, brutos = [], [], {}, {}
    total, feito = len(cen_def) * (len(pols) + tem_prog), 0
    for cen, cd in cen_def.items():
        d, c = cd["distribuicao"], cd["choque"]
        # mesmos futuros (números aleatórios comuns) para todas as políticas e para os dois HB
        rng = np.random.default_rng(_semente(par["semente"], prod["nome"], d, c))
        D, Vr, S = gerar_cenarios(prod["previsao"], d, int(par["n_rep"]), rng, cv, por_h, traj, par["mult_incerteza"],
                                  c, par["choque_inicio"], par["choque_fim"])
        p_hb = dict(prod, hb_dias=cd["hb_dias"])
        p2 = dict(par, _dist=d)
        for pol in pols + (["PROG"] if tem_prog else []):
            res = simular(p_hb, pol, D, Vr, S, p2, cv, z)
            r = res["rep"]
            reps.append(r.assign(cenario=cen, politica=pol, repeticao=np.arange(len(r))))
            mensal[(cen, pol)] = res["mensal"]
            brutos[(cen, pol)] = res["_bruto"]
            resumo.append({"cenario": cen, "distribuicao": d, "choque": c, "hb_dias": cd["hb_dias"], "politica": pol,
                           "nivel_servico": r["nivel_servico"].mean(), "nivel_garantido": r["nivel_servico"].quantile(0.05),
                           "meses_falta": r["meses_falta"].mean(), "meses_cob_baixa": r["meses_cob_baixa"].mean(),
                           "meses_excesso": r["meses_excesso"].mean(), "estoque_medio": r["estoque_medio"].mean(),
                           "estoque_final": r["estoque_final"].mean(), "reserva_media": r["reserva_media"].mean(),
                           "custo_estoque": r["custo_estoque"].mean(), "custo_falta": r["custo_falta"].mean(),
                           "custo_sobra": r["custo_sobra"].mean(), "custo_excesso": r["custo_excesso"].mean(),
                           "custo_total": r["custo_total"].mean(), "custo_p95": r["custo_total"].quantile(0.95),
                           "unidades_cortadas": r["unidades_cortadas"].mean()})
            feito += 1
            if progresso:
                progresso(feito / total, f"{cen} · {NOME_POLITICA[pol]}")
        brutos[(cen, "_cen")] = {"D": D, "S": S}
    resumo = pd.DataFrame(resumo)
    resumo["cumpre_metas"] = (resumo["nivel_servico"] >= par["meta_servico"]) & (resumo["nivel_garantido"] >= par["meta_garantido"])
    agora = datetime.now()
    return {"id": agora.strftime("%Y%m%d_%H%M%S"), "data": agora.strftime("%d/%m/%Y %H:%M"),
            "produto": prod, "parametros": {k: v for k, v in par.items() if not k.startswith("_")},
            "cenarios": cen_def, "z": z, "curva": curva.to_dict("list"), "resumo": resumo,
            "melhor": escolher(resumo, prod), "repeticoes": pd.concat(reps, ignore_index=True),
            "mensal": mensal, "_brutos": brutos, "stockpyl": None}


def escolher(resumo, prod):
    """Regra de escolha: entre as políticas que cumprem as metas, a de menor custo; se nenhuma cumpre, a de maior nível garantido.
    Com lote mínimo, a execução usa a versão em lotes (P2 → P2 + lote; P3 → P4)."""
    linhas = []
    for cen, g in resumo[resumo["politica"] != "PROG"].groupby("cenario", sort=False):
        ok = g[g["cumpre_metas"]]
        b = ok.sort_values("custo_total").iloc[0] if len(ok) else g.sort_values(["nivel_garantido", "custo_total"], ascending=[False, True]).iloc[0]
        impl = b["politica"]
        if int(prod.get("lote") or 0) > 0:
            impl = {"P2": "P2L", "P3": "P4"}.get(impl, impl)
        if impl not in set(g["politica"]):
            impl = b["politica"]
        p2 = g[g["politica"] == "P2"]
        prog = resumo[(resumo["cenario"] == cen) & (resumo["politica"] == "PROG")]
        linhas.append({"cenario": cen, "distribuicao": b["distribuicao"], "choque": b["choque"], "hb_dias": b["hb_dias"],
                       "politica": b["politica"], "como_executar": impl,
                       "cumpre_metas": bool(len(ok)), "nivel_servico": b["nivel_servico"], "nivel_garantido": b["nivel_garantido"],
                       "meses_falta": b["meses_falta"], "meses_excesso": b["meses_excesso"], "custo_total": b["custo_total"],
                       "custo_vs_P2": (b["custo_total"] / p2["custo_total"].iloc[0] - 1) if len(p2) else np.nan,
                       "custo_vs_programado": (b["custo_total"] / prog["custo_total"].iloc[0] - 1) if len(prog) else np.nan,
                       "motivo": "menor custo entre as que cumprem as metas" if len(ok) else
                                 "nenhuma cumpre as metas: maior nível garantido (levar ao S&OP)"})
    return pd.DataFrame(linhas)


# =====================================================================================================
# 8. Plano mês a mês e gatilhos de S&OE
# =====================================================================================================
def plano_deterministico(prod, pol, par, z, curva_cv):
    """Plano usando a previsão cadastrada como se fosse se confirmar (1 futuro, sem revisão, sem choque)."""
    F = np.asarray(prod["previsao"], float)
    n = len(F)
    D, S = F[None, :].copy(), np.zeros((1, n), bool)
    Vr = np.broadcast_to(F, (1, n, n)).copy()
    res = simular(prod, pol, D, Vr, S, dict(par, _dist="Normal"), curva_cv, z)
    return res["_bruto"]


TEXTO_ACAO = {
    "congelado_crit": "Dentro do horizonte congelado: não alterar a produção; acelerar entregas, realocar estoque e priorizar canais.",
    "congelado": "Dentro do horizonte congelado: acompanhar a execução; não mudar o plano sem exceção confirmada.",
    "aumento_tarde": "Só dá para reduzir neste mês (aumento fora do prazo de insumo/produção): buscar alternativas — antecipar "
                     "produção de outro mês, compra urgente de insumo, realocação entre canais.",
    "curto_up": "Curto prazo: aumentar ou antecipar produção, confirmar lote, garantir insumos críticos e capacidade.",
    "curto_down": "Curto prazo: adiar ou reduzir produção, suspender compras de insumo evitáveis e checar risco de E&O.",
    "curto_ok": "Curto prazo: manter o plano e revisar o próximo slot de produção e os compromissos de insumo.",
    "tatico_up": "Tático: rebalancear capacidade, antecipar componentes críticos e atualizar o cenário de alta no S&OP.",
    "tatico_down": "Tático: cancelar ou adiar insumos reversíveis, reduzir lotes futuros e atualizar o cenário de baixa / E&O.",
    "tatico_ok": "Tático: preservar flexibilidade; atualizar premissas do cenário e compromissos com fornecedores.",
    "medio": "Médio prazo: revisar premissas do lançamento, metas de S&OP e parâmetros da política; sem intervenção emergencial.",
}


def plano_e_acoes(resultado, cen, pol):
    """Tabela mês a mês: plano recomendado × programado, prazos, riscos (dos futuros simulados) e gatilho."""
    cd = resultado["cenarios"][cen]
    prod = dict(resultado["produto"], hb_dias=cd["hb_dias"])
    par, z = resultado["parametros"], resultado["z"]
    up, down = cd["janela_aumento"], cd["janela_reducao"]
    cv = cv_por_horizonte(pd.DataFrame(resultado["curva"]), len(prod["previsao"]) + 40)
    F = np.asarray(prod["previsao"], float)
    prog = np.asarray(prod["programada"], float)
    n, meses, lote = len(F), prod["meses"], int(prod.get("lote") or 0)
    rec = plano_deterministico(prod, pol, par, z, cv)
    proj = plano_deterministico(prod, "PROG", par, z, cv)
    risco = resultado["mensal"].get((cen, "PROG"))
    if risco is None:
        risco = resultado["mensal"].get((cen, pol))
    media = F[F > 0].mean() if (F > 0).any() else 0
    linhas = []
    for m in range(n):
        r_ch, p = rec["chegada_final"][0, m], prog[m]
        est_prog, pend_prog = proj["estoque"][0, m], proj["pendente"][0, m]
        cob_prog = proj["cob"][0, m]
        rf = float(risco["prob_falta"].iloc[m]) if risco is not None else np.nan
        rx = float(risco["prob_excesso"].iloc[m]) if risco is not None else np.nan
        dif = r_ch - p
        tol = max(GATILHO["tolerancia_manter"] * max(p, r_ch), lote * 0.5, 0.02 * media)
        if m < down:
            acao, dirc = "Congelado", "0"
        elif dif > tol:
            acao, dirc = ("Produzir mais", "up") if m >= up else ("Aumento fora do prazo", "up_late")
        elif dif < -tol:
            acao, dirc = "Reduzir", "down"
        else:
            acao, dirc = "Manter", "0"
        # gatilho (previsão de vendas = 0 → nenhum gatilho)
        motivos, sev = [], "OK"
        if F[m] <= 0:
            sev = "—"
        else:
            if pend_prog > 0:
                motivos.append("falta projetada com o plano programado")
            if np.isfinite(cob_prog) and cob_prog < par["cob_min"]:
                motivos.append(f"cobertura abaixo de {par['cob_min']} dias")
            if np.isfinite(rf) and rf >= GATILHO["risco_critico"]:
                motivos.append(f"risco de falta ≥ {int(100 * GATILHO['risco_critico'])}%")
            if motivos:
                sev = "Crítico"
            else:
                if np.isfinite(rf) and rf >= GATILHO["risco_atencao"]:
                    motivos.append(f"risco de falta entre {int(100 * GATILHO['risco_atencao'])}% e {int(100 * GATILHO['risco_critico'])}%")
                if (np.isfinite(cob_prog) and cob_prog > par["cob_max"]) or (np.isfinite(rx) and rx >= GATILHO["risco_excesso_atencao"]):
                    motivos.append(f"risco de excesso (cobertura > {par['cob_max']} dias)")
                if m >= down and acao != "Manter" and abs(dif) > GATILHO["desvio_atencao"] * max(p, 1):
                    motivos.append(f"programado difere {int(100 * GATILHO['desvio_atencao'])}% ou mais do recomendado")
                if motivos:
                    sev = "Atenção"
        if m < down:
            txt = TEXTO_ACAO["congelado_crit" if sev == "Crítico" else "congelado"]
        elif dirc == "up_late" or (m < up and sev == "Crítico"):
            txt = TEXTO_ACAO["aumento_tarde"]
        elif m <= 3:
            txt = TEXTO_ACAO["curto_up" if dirc == "up" else "curto_down" if dirc == "down" else "curto_ok"]
        elif m <= 6 or dirc in ("up", "down"):
            txt = TEXTO_ACAO["tatico_up" if dirc == "up" else "tatico_down" if dirc == "down" else "tatico_ok"]
        else:
            txt = TEXTO_ACAO["medio"]
        linhas.append({"mes": meses[m], "previsao": F[m], "programada": p, "recomendada": r_ch, "acao": acao, "diferenca": dif,
                       "decidir_aumento_ate": meses[m - up] if m - up >= 0 else "prazo vencido",
                       "decidir_reducao_ate": meses[m - down] if m - down >= 0 else "congelado",
                       "estoque_proj_programado": est_prog - pend_prog, "cobertura_programado": cob_prog if F[m] > 0 else np.nan,
                       "estoque_proj_recomendado": rec["estoque"][0, m] - rec["pendente"][0, m],
                       "reserva": rec["ss"][0, m], "risco_falta": rf, "risco_excesso": rx, "gatilho": sev,
                       "motivo": "; ".join(motivos), "o_que_fazer": txt if (F[m] > 0 or p > 0 or r_ch > 0) else "",
                       "_congelado": m < down})
    df = pd.DataFrame(linhas)
    visivel = ~((df["previsao"] <= 0) & df["_congelado"])          # previsão zero dentro do HB não aparece
    return df[visivel].drop(columns="_congelado").reset_index(drop=True)


# =====================================================================================================
# 9. Segunda opinião: série parecida no experimento anterior
# =====================================================================================================
def carregar_referencia():
    for p in ARQUIVOS_REFERENCIA:
        try:
            if p.exists():
                return json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            pass
    return None


def perfil(prev):
    v = np.array([x for x in prev if x > 0], float)
    if len(v) < 2:
        return {"media": float(v.mean()) if len(v) else 0.0, "cv": 0.0, "pico": 0.0, "n": len(v)}
    return {"media": float(v.mean()), "cv": float(v.std(ddof=1) / v.mean()), "pico": float(v.max() / v.mean()), "n": len(v)}


def segunda_opiniao(prod, ref, dist, choque, hb):
    """Série mais parecida entre as 8 testadas no experimento anterior (origem, cobertura, LT, volume, variação, pico).
    Semelhança = 100·e^(−distância/3). A política vem do experimento anterior (janela simples, distribuição Normal/Poisson)."""
    if not ref:
        return None
    prof = pd.DataFrame(ref["profiles"])
    sku = prod.get("sku_base")
    if sku in set(prof["sku"]):          # série da base: usa o mesmo perfil do experimento e não compara com ela mesma
        row = prof[prof["sku"] == sku].iloc[0]
        pf = {"media": row["forecast_mean"], "cv": row["forecast_cv"], "pico": row["forecast_peak_ratio"]}
        prof = prof[prof["sku"] != sku]
    else:
        pf = perfil(prod["previsao"])
    feats = {"cobertura": ("target_days", float(prod["cobertura_dias"]), 1.0),
             "lt": ("max_material_lt_weeks", float(prod["lt_insumo_semanas"]), 1.0),
             "volume": ("forecast_mean", math.log10(max(pf["media"], 1)), 1.0),
             "cv": ("forecast_cv", pf["cv"], 1.5), "pico": ("forecast_peak_ratio", pf["pico"], 1.0)}
    X = prof.copy()
    X["forecast_mean"] = np.log10(X["forecast_mean"].clip(lower=1))
    d2 = (X["origin"] != prod["origem"]).astype(float) ** 2
    todos = pd.DataFrame(ref["profiles"]).assign(forecast_mean=lambda d: np.log10(d["forecast_mean"].clip(lower=1)))
    for colname, val, w in feats.values():
        sd = todos[colname].std(ddof=1) or 1.0
        d2 = d2 + w * ((val - X[colname]) / sd) ** 2
    X["distancia"] = np.sqrt(d2)
    X["semelhanca"] = 100 * np.exp(-X["distancia"] / 3)
    X = X.sort_values("distancia")
    hb_ref = 60 if float(hb) > 45 else 30
    dref = "Normal" if dist.startswith("Erros") else dist
    best = pd.DataFrame(ref["best"])

    def pol_de(sku):
        b = best[(best["sku"] == sku) & (best["hb_days"] == hb_ref) & (best["distribution"] == dref) & (best["shock"] == choque)]
        return (b["best_policy"].iloc[0], bool(b["best_meets_floors"].iloc[0])) if len(b) else ("—", False)

    top = [{"serie": s, "semelhanca": sim, "politica": pol_de(s)[0], "cumpre": pol_de(s)[1]}
           for s, sim in zip(X["sku"].head(3), X["semelhanca"].head(3))]
    votos = pd.Series([t["politica"] for t in top]).value_counts()
    ranking = X[["sku", "kind", "origin", "target_days", "max_material_lt_weeks", "forecast_cv", "forecast_peak_ratio", "semelhanca"]]
    proprio = pol_de(sku) if sku in set(pd.DataFrame(ref["profiles"])["sku"]) else None
    return {"top": top, "ranking": ranking, "hb": hb_ref, "dist_ref": dref, "proprio": proprio,
            "confianca": "alta" if votos.iloc[0] == 3 else "média" if votos.iloc[0] == 2 else "baixa"}


# =====================================================================================================
# 10. Confirmação no Stockpyl: executa no Stockpyl as mesmas decisões do SOLARIS e compara
# =====================================================================================================
def confirmar_stockpyl(resultado, n_rep, progresso=None):
    """Para as primeiras n_rep repetições de cada cenário e política, roda o Stockpyl com as chegadas decididas pelo SOLARIS.
    O Stockpyl usa prazo de entrega = janela de redução (o pedido vira firme nesse momento); os meses iniciais entram como
    um aquecimento sem demanda, para incluir a produção já programada. O choque usa o DisruptionProcess (retém o recebimento)."""
    from stockpyl.supply_chain_network import single_stage_system
    from stockpyl.sim import initialize, step, close
    from stockpyl.disruption_process import DisruptionProcess
    prod, par = resultado["produto"], resultado["parametros"]
    preco = float(prod["preco"])
    chaves = [k for k in resultado["_brutos"] if k[1] not in ("_cen", "PROG")]
    linhas, total, feito = [], len(chaves) * n_rep, 0
    for (cen, pol) in chaves:
        bru, sc = resultado["_brutos"][(cen, pol)], resultado["_brutos"][(cen, "_cen")]
        W = int(resultado["cenarios"][cen]["janela_reducao"])
        for r in range(min(n_rep, sc["D"].shape[0])):
            Dr, Sr, ch = sc["D"][r], sc["S"][r], bru["chegada_final"][r]
            n = len(Dr)
            dem = [0.0] * W + [float(x) for x in Dr]
            net = single_stage_system(holding_cost=par["custo_estoque_aa"] / 12 * preco, stockout_cost=par["custo_falta"] * preco,
                                      demand_type="D", demand_list=dem, policy_type="BS", base_stock_level=0.0,
                                      shipment_lead_time=W, initial_inventory_level=float(prod["estoque_inicial"]))
            node = net.nodes[0]
            estados = [False] * W + [bool(x) for x in Sr]
            if any(estados):
                node.disruption_process = DisruptionProcess(random_process_type="E", disruption_type="RP",
                                                            disruption_state_list=estados)
            initialize(network=net, num_periods=W + n, rand_seed=0)
            ip = float(prod["estoque_inicial"])
            for k in range(W + n):
                q = float(ch[k]) if k < n else 0.0           # pedido no período k chega em k + W (= mês k da simulação)
                ip_dep = ip - dem[k]
                node.inventory_policy.base_stock_level = ip_dep + q
                step(network=net)
                ip = ip_dep + q
            close(network=net)
            il = np.array([float(list(node.state_vars[W + t].inventory_level.values())[0]) for t in range(n)])
            est_spl = np.maximum(il, 0.0)
            dif = float(np.max(np.abs(est_spl - bru["estoque"][r])))
            aval = np.asarray(prod["previsao"]) > 0
            atend = np.minimum(Dr, np.maximum(il + Dr, 0.0))          # atendido = demanda − aumento da pendência
            linhas.append({"cenario": cen, "politica": pol, "repeticao": r,
                           "estoque_medio_solaris": float(bru["estoque"][r][aval].mean()) if aval.any() else 0.0,
                           "estoque_medio_stockpyl": float(est_spl[aval].mean()) if aval.any() else 0.0,
                           "pendencia_final_solaris": float(bru["pendente"][r][-1]), "pendencia_final_stockpyl": float(max(-il[-1], 0.0)),
                           "maior_diferenca_un": dif, "identico": dif < 1e-3 * max(1.0, float(np.max(Dr)))})
            feito += 1
            if progresso:
                progresso(feito / max(total, 1), f"Stockpyl · {cen} · {pol}")
    return pd.DataFrame(linhas)


# =====================================================================================================
# 11. Histórico: salvar, listar, carregar e exportar para Excel
# =====================================================================================================
def _serializavel(resultado):
    r = {k: v for k, v in resultado.items() if not k.startswith("_") and k not in ("repeticoes", "mensal", "resumo", "melhor", "stockpyl")}
    r["resumo"] = resultado["resumo"].to_dict("records")
    r["melhor"] = resultado["melhor"].to_dict("records")
    r["mensal"] = [{"cenario": cen, "politica": p, **{k: [float(x) for x in v] for k, v in df.to_dict("list").items()}}
                   for (cen, p), df in resultado["mensal"].items()]
    r["stockpyl"] = resultado["stockpyl"].to_dict("records") if resultado.get("stockpyl") is not None else None
    return r


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, (np.bool_,)):
        return bool(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def excel_resultado(resultado):
    """Excel da rodada: parâmetros, produto, melhor política, resumo, plano e ações, risco mensal, Stockpyl e repetições."""
    prod = resultado["produto"]
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as w:
        pd.DataFrame([{"parâmetro": k, "valor": str(v)} for k, v in resultado["parametros"].items()] +
                     [{"parâmetro": "z", "valor": resultado["z"]}, {"parâmetro": "data", "valor": resultado["data"]}]
                     ).to_excel(w, sheet_name="Parametros", index=False)
        pd.DataFrame([{"cenario": k, **v} for k, v in resultado["cenarios"].items()]).to_excel(w, sheet_name="Cenarios", index=False)
        pd.DataFrame([{c: prod.get(c) for c in CAMPOS}]).to_excel(w, sheet_name="Produto", index=False)
        pd.DataFrame({"mes": prod["meses"], "previsao": prod["previsao"], "programada": prod["programada"]}).to_excel(
            w, sheet_name="Produto", index=False, startrow=3)
        resultado["melhor"].to_excel(w, sheet_name="Melhor_politica", index=False)
        resultado["resumo"].to_excel(w, sheet_name="Resumo", index=False)
        for i, b in resultado["melhor"].reset_index(drop=True).iterrows():
            try:
                plano_e_acoes(resultado, b["cenario"], b["como_executar"]).assign(cenario=b["cenario"], politica=b["como_executar"]
                                                                                    ).to_excel(w, sheet_name=f"Acao_{i + 1}", index=False)
            except Exception:
                pass
        pd.concat([df.assign(mes=prod["meses"], cenario=cen, politica=p) for (cen, p), df in resultado["mensal"].items()]
                  ).to_excel(w, sheet_name="Risco_mensal", index=False)
        if resultado.get("stockpyl") is not None:
            resultado["stockpyl"].to_excel(w, sheet_name="Validacao_Stockpyl", index=False)
        rep = resultado.get("repeticoes")
        if rep is not None and len(rep) <= 20_000:
            rep.to_excel(w, sheet_name="Repeticoes", index=False)
        elif rep is not None:
            pd.DataFrame({"aviso": [f"{len(rep)} linhas: as repetições estão no arquivo _repeticoes.csv da mesma rodada."]}
                         ).to_excel(w, sheet_name="Repeticoes", index=False)
    return buf.getvalue()


def bytes_excel(resultado):
    """Excel da rodada, gerado uma vez e guardado no próprio resultado (evita refazer a cada clique)."""
    if resultado.get("_xlsx") is None:
        resultado["_xlsx"] = excel_resultado(resultado)
    return resultado["_xlsx"]


def json_resultado(resultado):
    return json.dumps(_serializavel(resultado), ensure_ascii=False, default=_json_default).encode("utf-8")


def salvar_resultado(resultado):
    PASTA_HIST.mkdir(parents=True, exist_ok=True)
    nome = f"{resultado['id']}_{norm_txt(resultado['produto']['nome'])[:30]}"
    (PASTA_HIST / f"{nome}.json").write_bytes(json_resultado(resultado))
    (PASTA_HIST / f"{nome}.xlsx").write_bytes(bytes_excel(resultado))
    if resultado.get("repeticoes") is not None:
        resultado["repeticoes"].to_csv(PASTA_HIST / f"{nome}_repeticoes.csv", index=False, sep=";", decimal=",")
    return nome


def listar_historico():
    if not PASTA_HIST.exists():
        return pd.DataFrame()
    linhas = []
    for f in sorted(PASTA_HIST.glob("*.json"), reverse=True):
        try:
            r = json.loads(f.read_text(encoding="utf-8"))
            for b in r["melhor"]:
                linhas.append({"arquivo": f.stem, "data": r["data"], "produto": r["produto"]["nome"],
                               "simulacoes": r["parametros"]["n_rep"], "protecao": r["parametros"]["protecao"],
                               "cenario": b["cenario"], "politica": b["politica"], "como_executar": b["como_executar"],
                               "cumpre_metas": b["cumpre_metas"], "nivel_servico": b["nivel_servico"],
                               "nivel_garantido": b["nivel_garantido"], "custo_total": b["custo_total"],
                               "stockpyl": r.get("stockpyl") is not None})
        except Exception:
            continue
    return pd.DataFrame(linhas)


def carregar_resultado(nome):
    """Recarrega uma rodada salva (sem as repetições brutas) para consultar nas telas."""
    r = json.loads((PASTA_HIST / f"{nome}.json").read_text(encoding="utf-8"))
    r["resumo"] = pd.DataFrame(r["resumo"])
    r["melhor"] = pd.DataFrame(r["melhor"])
    r["mensal"] = {(m["cenario"], m["politica"]): pd.DataFrame({k: v for k, v in m.items() if k not in ("cenario", "politica")})
                   for m in r["mensal"]}
    r["stockpyl"] = pd.DataFrame(r["stockpyl"]) if r.get("stockpyl") else None
    r["repeticoes"] = None
    r["_brutos"] = {}
    r["_arquivo"] = nome
    r["_xlsx"] = bytes_historico(nome)
    return r


def apagar_resultado(nome):
    for ext in (".json", ".xlsx", "_repeticoes.csv"):
        p = PASTA_HIST / f"{nome}{ext}"
        if p.exists():
            p.unlink()


def bytes_historico(nome, ext=".xlsx"):
    p = PASTA_HIST / f"{nome}{ext}"
    return p.read_bytes() if p.exists() else None


def zip_historico():
    import zipfile
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(PASTA_HIST.glob("*")) if PASTA_HIST.exists() else []:
            z.write(f, f.name)
    return buf.getvalue()


# =====================================================================================================
# 12. Backtest com o histórico real (só produtos da base): o que cada política teria feito com as versões reais
# =====================================================================================================
def backtest_historico(serie, par, z, curva, versao_idx=0):
    """Backtest com o histórico real: a demanda é o realizado e a previsão revisada é a de cada versão real do triângulo.
    P0 = o que a empresa de fato produziu. Só faz sentido para séries com realizado (benchmarks e produtos com histórico)."""
    r0 = int(versao_idx)
    T = serie["T_real"]
    if T - r0 < 3 or (serie["realizado"][r0:T] > 0).sum() < 3:
        return None
    A = np.nan_to_num(serie["realizado"][r0:T])
    n = len(A)
    V = np.nan_to_num(serie["V"][r0:T, r0:T])
    prod = produto_da_base(serie, r0)
    prod = dict(prod, previsao=[float(x) for x in np.nan_to_num(serie["V"][r0, r0:T])],
                programada=[float(x) for x in np.nan_to_num(serie["producao_real"][r0:T])], meses=serie["meses"][r0:T])
    Vr = np.zeros((1, n, n))
    for t in range(n):
        Vr[0, t, t:] = V[t, t:]
    Vr[0, 0, :] = prod["previsao"]
    D, S = A[None, :].astype(float), np.zeros((1, n), bool)
    cv = cv_por_horizonte(curva, n + 40)
    linhas = []
    for pol in ["PROG"] + [p for p in par["politicas"] if p != "P2L" or int(prod.get("lote") or 0) > 0]:
        res = simular(prod, pol, D, Vr, S, dict(par, _dist="Normal", _aval=(A > 0) | (np.asarray(prod["previsao"]) > 0)), cv, z)
        rr = res["rep"].iloc[0]
        linhas.append({"politica": "P0" if pol == "PROG" else pol, "nivel_servico": rr["nivel_servico"], "meses_falta": rr["meses_falta"],
                       "meses_excesso": rr["meses_excesso"], "estoque_medio": rr["estoque_medio"], "estoque_final": rr["estoque_final"],
                       "custo_total": rr["custo_total"],
                       "_estoque": res["_bruto"]["estoque"][0] - res["_bruto"]["pendente"][0]})
    return {"tabela": pd.DataFrame(linhas), "meses": prod["meses"], "realizado": A, "previsao_inicial": prod["previsao"]}

Writing motor.py


## Passo 4 — Testar o motor no notebook (antes de abrir o app)

Se o app mostrar outro número para a mesma configuração e semente, o erro está no app.

In [ ]:
import importlib, motor
importlib.reload(motor)  # se você editar o motor.py, o reload carrega a versão nova

dados = motor.carregar_base(BASE)
display(pd.DataFrame([{"série": k, "tipo": s["tipo"], "origem": s["origem"], "cobertura (dias)": s["cobertura_dias"],
                       "HB (dias)": s["hb_dias"], "LT insumo (sem.)": s["lt_insumo_semanas"], "lote": s["lote"],
                       "versões": len(s["versoes"]), "meses realizados": s["T_real"]} for k, s in dados["series"].items()]))
display(dados["curva"].round(3).T)  # erro da previsão por horizonte (CV), calibrado com os benchmarks

,série,tipo,origem,cobertura (dias),HB (dias),LT insumo (sem.),lote,versões,meses realizados
0,PRODUTO 1,produto,Terceiro,70.0,60.0,16.0,57508,14,13
1,BENCH A,benchmark,Terceiro,70.0,60.0,16.0,0,34,33
2,BENCH B,benchmark,Terceiro,70.0,60.0,16.0,28856,24,23
3,PRODUTO 2,produto,Interno,90.0,30.0,10.0,0,10,9
4,BENCH C,benchmark,Interno,90.0,30.0,10.0,39766,28,27
5,PRODUTO 3,produto,Interno,59.0,30.0,8.0,45926,14,13
6,BENCH D,benchmark,Interno,59.0,30.0,8.0,48323,34,33
7,BENCH E,benchmark,Interno,59.0,30.0,8.0,0,24,23


,0,1,2,3,4,5,6,7,8,9,10,11,12
h,0.000,1.000,2.000,3.000,4.000,5.000,6.000,7.0,8.0,9.000,10.000,11.000,12.000
cv,0.607,0.619,0.691,0.735,0.902,0.879,0.988,1.0,1.0,0.945,0.893,0.874,0.904
n,113.000,113.000,113.000,113.000,111.000,109.000,107.000,104.0,99.0,94.000,89.000,84.000,79.000


In [ ]:
# Teste 1 — uma rodada pequena do PRODUTO 1 na versão do lançamento (300 futuros, com e sem choque).
produto = motor.produto_da_base(dados["series"]["PRODUTO 1"], 0)
par = dict(motor.PADRAO, n_rep=300, choques=["Sem choque", "Com choque"])
print("Janelas (aumentar, reduzir) em meses:", motor.janelas(produto, par))
t0 = time.time()
res = motor.rodar(produto, par, dados)
print(f"Rodada em {time.time() - t0:.1f} s")
display(res["melhor"][["cenario", "politica", "como_executar", "cumpre_metas", "nivel_servico", "nivel_garantido", "custo_total"]])

In [ ]:
# Teste 2 — plano de ação do primeiro cenário e confirmação no Stockpyl (as trajetórias devem ser idênticas).
cen = list(res["cenarios"])[0]
display(motor.plano_e_acoes(res, cen, res["melhor"].iloc[0]["como_executar"])[
    ["mes", "previsao", "programada", "recomendada", "acao", "decidir_aumento_ate", "gatilho"]])
try:
    v = motor.confirmar_stockpyl(res, 10)
    print("Stockpyl: trajetórias idênticas em", f"{v['identico'].mean():.0%}", "| maior diferença:", round(v["maior_diferenca_un"].max(), 6), "un.")
except ImportError:
    print("Stockpyl não instalado.")

## Passo 5 — Textos da tela de Início: `conteudo.py`
Edite os textos aqui (o que é a ferramenta, como usar, equipe, documentos). Depois de regravar, recarregue a página.

In [ ]:
%%writefile conteudo.py
# conteudo.py — textos da tela de Início. Edite à vontade: o app lê tudo daqui.
# Dica: texto entre ** fica em negrito; cada item de lista é uma string.

TITULO = "SOLARIS · Estoque de lançamentos"
SUBTITULO = "Ferramenta de apoio à decisão de produção e estoque para lançamentos de produtos"

O_QUE_E = (
    "O SOLARIS recomenda **qual política de estoque usar** em um lançamento e **o que fazer mês a mês** no S&OE. "
    "Para isso, ele simula centenas de futuros possíveis (Monte Carlo) para a demanda do produto, aplica as políticas "
    "P2, P3 e P4 em cada futuro, com as mesmas regras de prazo da operação (horizonte congelado, lead time de insumo e de "
    "produção), e compara nível de serviço, cobertura e custo."
)

PROBLEMA = (
    "Lançamentos não têm histórico de vendas, e a previsão erra muito nos primeiros meses. Produzir de menos gera ruptura "
    "na fase mais importante do produto; produzir demais gera excesso e risco de descarte. A ferramenta mostra esse "
    "equilíbrio com números e transforma a recomendação em ações com prazo."
)

COMO_USAR = [
    "**Escolha o produto** na barra lateral: um produto da base (e a versão da previsão) ou um produto cadastrado.",
    "**Cadastre produtos novos** na aba Cadastro: preencha à mão ou suba a planilha modelo.",
    "**Escolha as análises de sensibilidade**: distribuição da demanda, com ou sem choque de fornecimento, HB, fator de "
    "segurança e número de simulações. Os custos ficam visíveis e já vêm com os valores usados no projeto.",
    "**Clique em Rodar simulação.** A rodada é salva automaticamente no Histórico (Google Drive).",
    "**Leia a Recomendação**, depois a **Ação S&OE** (o que fazer em cada mês e até quando decidir) e, se quiser, a "
    "**Análise avançada** com os gráficos.",
]

POLITICAS = {
    "P2 · Repor o previsto": "Todo mês produz o previsto até a produção chegar mais a cobertura-alvo, descontando o estoque e "
                             "o que já está a caminho. Não tem reserva para o erro da previsão.",
    "P3 · Previsto + reserva": "Igual ao P2, com uma reserva de segurança (z × σ) dimensionada pelo erro histórico da previsão "
                               "em cada horizonte.",
    "P4 · Reserva em lotes": "Igual ao P3, com a produção arredondada para múltiplos do lote mínimo do produto.",
    "Plano programado": "O plano de produção que já está no sistema. Entra como referência para comparar e para calcular "
                        "os riscos da Ação S&OE.",
}

JANELA = (
    "**Janela de reação.** Para **aumentar** a produção de um mês, a decisão precisa ser tomada com "
    "max(HB, LT do insumo) + 1 mês de produção + 1 mês de decisão de antecedência. Para **reduzir**, basta "
    "HB + 1 mês de decisão (não depende do insumo nem da produção). Dentro do HB o plano está congelado."
)

METRICAS = {
    "Nível de serviço": "Parte da demanda atendida no próprio mês (a falta vira pendência).",
    "Nível garantido": "Nível de serviço em 95% dos futuros simulados (percentil 5). Mede o pior caso razoável.",
    "% meses com falta": "Meses em que faltou produto em relação aos meses com previsão de venda.",
    "Cobertura": "Dias que o estoque dura consumindo a previsão dos meses seguintes. Abaixo de 30 dias = cobertura baixa; "
                 "acima de 180 dias = excesso.",
    "Custo total": "Valores-padrão, editáveis na barra lateral: custo de manter estoque (25% a.a. sobre o preço) + falta (1 × preço por unidade pendente por mês) + sobra "
                   "no fim (0,3 × preço) + excesso acima de 6 meses de previsão (0,05 × preço por mês).",
    "Meses sem previsão": "Meses com previsão de venda igual a zero (pré-lançamento) ficam fora das métricas de avaliação e "
                          "não acionam gatilho.",
}

EQUIPE = {
    "Autoras": ["Betina Moller", "Julia Jallad"],
    "Orientador": ["Prof. Rodrigo Caiado"],
    "Apoio": ["L'Oréal Brasil", "PUC-Rio · Departamento de Engenharia Industrial"],
}

PROJETO = "Projeto de graduação (TCC) · Engenharia de Produção · PUC-Rio · 2026"

AGRADECIMENTOS = (
    "Agradecemos à equipe de Supply Chain da L'Oréal Brasil pelos dados e pelas discussões sobre o processo de S&OE, "
    "e ao Prof. Rodrigo Caiado pela orientação."
)

# Documentos para download na tela de Início: coloque os arquivos na pasta docs/ dentro da pasta do projeto.
# Chave = nome do arquivo; valor = descrição mostrada na tela. Arquivos da pasta que não estiverem aqui aparecem com o próprio nome.
DOCUMENTOS = {
    "SOLARIS_Manual_do_usuario.pdf": "Manual do usuário da ferramenta",
    "SOLARIS_Documentacao_Codigo_e_Resultados.pdf": "Documentação do código e dos resultados do experimento",
    "SOLARIS_Metodologia.docx": "Metodologia detalhada",
    "SOLARIS_Relatorio_Metodologia_Resultados.docx": "Capítulos de metodologia e resultados do relatório",
}

AVISO_DADOS = (
    "Os dados da base são confidenciais e foram cedidos para o projeto. Não compartilhe o link público do túnel fora da equipe."
)

## Passo 6 — Esqueleto do app: `app.py`

Lê a base (com cache), monta a barra lateral e as abas, roda a simulação quando o botão é clicado e salva no histórico.
Comandos novos em relação ao notebook da disciplina: `st.session_state` (guarda o resultado entre cliques),
`st.progress` (barra de progresso), `st.toggle`, `st.number_input`, `st.expander` na barra lateral.

In [ ]:
%%writefile app.py
# app.py — esqueleto do SOLARIS: barra lateral (produto, sensibilidade, custos), botão de rodar e 6 telas em abas.
# Cada tela mora em um arquivo próprio (tela_inicio.py, tela_cadastro.py, ...) com a mesma porta de entrada: mostrar(dados, f).
import importlib
import traceback

import streamlit as st

import motor  # toda a lógica: base, Monte Carlo, políticas, métricas, histórico (sem nenhum comando st.)

st.set_page_config(page_title="SOLARIS · Lançamentos", page_icon="☀️", layout="wide")
ss = st.session_state


@st.cache_data(show_spinner="Lendo a base")
def carregar(caminho, versao_arquivo):  # versao_arquivo = data de modificação: se a base mudar, lê de novo
    return motor.carregar_base(caminho)


# ---------- Base de dados ----------
if not motor.ARQUIVO_BASE.exists():
    st.title("SOLARIS")
    st.warning(f"Não encontrei a base **{motor.ARQUIVO_BASE.name}** na pasta do projeto ({motor.PASTA}). Envie o arquivo:")
    arq = st.file_uploader("Base (Triangulo Produtos V2.xlsx)", type=["xlsx"])
    if arq:
        motor.PASTA.mkdir(parents=True, exist_ok=True)
        motor.ARQUIVO_BASE.write_bytes(arq.getvalue())
        st.rerun()
    st.stop()

dados = carregar(str(motor.ARQUIVO_BASE), motor.ARQUIVO_BASE.stat().st_mtime)
ss.setdefault("cadastrados", motor.carregar_cadastrados())
ss.setdefault("resultado", None)
if "_selecionar" in ss:  # outra tela pediu para trocar o produto (ex.: acabou de cadastrar)
    ss["escolha_produto"] = ss.pop("_selecionar")

# ---------- Barra lateral: produto ----------
st.sidebar.header("1 · Produto")
series = dados["series"]
nomes_base = list(series)
nomes_cad = [f"✚ {p['nome']}" for p in ss["cadastrados"]]
opcoes = nomes_base + nomes_cad
if ss.get("escolha_produto") not in opcoes:
    ss["escolha_produto"] = opcoes[0]
escolha = st.sidebar.selectbox("Produto em análise", opcoes, key="escolha_produto",
                               help="Produtos e benchmarks da base, ou produtos cadastrados (✚) na aba Cadastro.")
if escolha in series:
    s = series[escolha]
    versoes = motor.versoes_validas(s)
    versao = st.sidebar.selectbox("Versão da previsão (mês “hoje”)", versoes, format_func=lambda i: s["versoes"][i],
                                  help="O mês em que a decisão é tomada: usa a previsão e o plano dessa versão e o estoque real "
                                       "do mês anterior. A primeira versão é o momento do lançamento.")
    produto = motor.produto_da_base(s, versao)
else:
    produto = dict(next(p for p in ss["cadastrados"] if f"✚ {p['nome']}" == escolha))
st.sidebar.caption(f"{produto['origem']} · cobertura-alvo {motor.num(produto['cobertura_dias'])} dias · HB "
                   f"{motor.num(produto['hb_dias'])} dias · LT insumo {motor.num(produto['lt_insumo_semanas'])} sem. · "
                   f"lote {motor.num(produto['lote']) if produto['lote'] else 'sem lote'}")

# ---------- Barra lateral: sensibilidade ----------
P = motor.PADRAO
st.sidebar.header("2 · Análises de sensibilidade")
distribuicoes = st.sidebar.multiselect(
    "Distribuição da demanda", motor.DISTRIBUICOES, default=P["distribuicoes"],
    help="Erros reais: a demanda é a previsão × (1 + um erro real dos benchmarks no mesmo horizonte). "
         "Normal e Poisson: variação em torno da previsão, para comparação.")
choques = st.sidebar.multiselect("Choque de fornecimento", motor.CHOQUES, default=P["choques"],
                                 help="Com choque: em cada mês há 10% de chance de começar uma interrupção (e 50% de terminar); "
                                      "durante o choque a produção não chega e fica retida.")
hb_prod = int(round(produto["hb_dias"]))
hbs = st.sidebar.multiselect("HB (dias)", sorted({30, 60, hb_prod}), default=[hb_prod],
                             help="Horizonte congelado. Escolha os dois para ver o efeito de mudar o HB.")
politicas = st.sidebar.multiselect("Políticas", motor.POLITICAS, default=P["politicas"], format_func=lambda p: motor.NOME_POLITICA[p],
                                   help="P2 + lote só roda quando o produto tem lote mínimo.")
protecao = st.sidebar.radio("Fator de segurança (proteção)", list(motor.PROTECAO_Z), horizontal=True,
                            help="Probabilidade de não faltar que a reserva de segurança mira: 95% (z = 1,645), 90% (z = 1,282) "
                                 "ou 85% (z = 1,036). Vale para P3 e P4.")
n_rep = st.sidebar.slider("Número de simulações (futuros por cenário)", 100, 2000, P["n_rep"], 100)

# ---------- Barra lateral: custos (visíveis, com os valores do projeto) ----------
st.sidebar.header("3 · Custos")
c1, c2 = st.sidebar.columns(2)
custo_estoque = c1.number_input("Estoque (% a.a.)", 0.0, 100.0, 100 * P["custo_estoque_aa"], 1.0,
                                help="Custo de manter estoque por ano, em % do preço (R$ PF).") / 100
custo_falta = c2.number_input("Falta (× PF)", 0.0, 10.0, P["custo_falta"], 0.1, help="Por unidade pendente por mês.")
custo_sobra = c1.number_input("Sobra final (× PF)", 0.0, 5.0, P["custo_sobra"], 0.05, help="Por unidade em estoque no fim do horizonte.")
custo_excesso = c2.number_input("Excesso (× PF/mês)", 0.0, 5.0, P["custo_excesso"], 0.01,
                                help="Por unidade acima de 6 meses de previsão, por mês.")

# ---------- Barra lateral: avançado ----------
with st.sidebar.expander("Avançado"):
    regra = st.radio("Regra da janela de reação", ["Assimétrica (validada)", "Simples (experimento anterior)"],
                     help="Assimétrica: aumentar = max(HB, LT insumo) + produção + decisão; reduzir = HB + decisão. "
                          "Simples: max(HB, 1 mês) nos dois sentidos.")
    mes_decisao = st.toggle("Somar 1 mês de decisão", value=P["mes_decisao"])
    lt_prod = st.number_input("LT de produção (meses)", 0, 6, P["lt_producao"])
    mult = st.slider("Multiplicador da incerteza", 0.5, 2.0, P["mult_incerteza"], 0.1,
                     help="1,0 = erro histórico. 1,5 = erros 50% maiores que os históricos.")
    a1, a2 = st.columns(2)
    p_ini = a1.number_input("Choque: início (%)", 0.0, 100.0, 100 * P["choque_inicio"], 1.0) / 100
    p_fim = a2.number_input("Choque: fim (%)", 1.0, 100.0, 100 * P["choque_fim"], 1.0) / 100
    meta_s = a1.number_input("Meta de serviço (%)", 50.0, 100.0, 100 * P["meta_servico"], 0.5) / 100
    meta_g = a2.number_input("Meta garantida P5 (%)", 50.0, 100.0, 100 * P["meta_garantido"], 0.5) / 100
    cob_min = a1.number_input("Cobertura baixa (< dias)", 0, 120, P["cob_min"], 5)
    cob_max = a2.number_input("Excesso (> dias)", 60, 720, P["cob_max"], 10)
    semente = st.number_input("Semente aleatória", 0, 10_000_000, P["semente"], 1,
                              help="Mesma semente = mesmos futuros: os resultados se repetem.")
    usar_spl = st.checkbox("Confirmar no Stockpyl (mais lento)", value=P["stockpyl"],
                           help="Executa as mesmas decisões no simulador Stockpyl e compara estoque e pendência.")
    n_spl = st.slider("Simulações no Stockpyl", 10, 200, P["n_rep_stockpyl"], 10, disabled=not usar_spl)

par = dict(P, n_rep=n_rep, protecao=protecao, politicas=politicas, distribuicoes=distribuicoes, choques=choques, hbs=hbs,
           custo_estoque_aa=custo_estoque, custo_falta=custo_falta, custo_sobra=custo_sobra, custo_excesso=custo_excesso,
           regra_janela=regra, mes_decisao=mes_decisao, lt_producao=lt_prod, mult_incerteza=mult, choque_inicio=p_ini,
           choque_fim=p_fim, meta_servico=meta_s, meta_garantido=meta_g, cob_min=cob_min, cob_max=cob_max, semente=semente,
           stockpyl=usar_spl, n_rep_stockpyl=n_spl)
up, down = motor.janelas(produto, par)
st.sidebar.info(f"Janela de reação: **aumentar** com {up} meses de antecedência · **reduzir** com {down}.")

faltando = [n for n, v in [("distribuição", distribuicoes), ("choque", choques), ("HB", hbs), ("política", politicas)] if not v]
erros = motor.validar_produto(produto)
rodar = st.sidebar.button("▶ Rodar simulação", type="primary", width="stretch", disabled=bool(faltando or erros))
if faltando:
    st.sidebar.error("Escolha pelo menos uma opção de: " + ", ".join(faltando) + ".")
if erros:
    st.sidebar.error(" ".join(erros))

st.title("☀️ SOLARIS · Estoque de lançamentos")

# ---------- Rodar ----------
if rodar:
    barra = st.progress(0.0, text="Simulando…")
    try:
        r = motor.rodar(produto, par, dados, progresso=lambda x, t: barra.progress(min(x, 1.0), text=t))
        if usar_spl:
            try:
                barra.progress(0.0, text="Confirmando no Stockpyl…")
                r["stockpyl"] = motor.confirmar_stockpyl(r, n_spl, progresso=lambda x, t: barra.progress(min(x, 1.0), text=t))
            except ImportError:
                st.warning("O Stockpyl não está instalado (pip install stockpyl). A rodada segue só com o simulador do SOLARIS.")
        ss["resultado"] = r
        barra.progress(1.0, text="Salvando no Histórico…")
        try:
            r["_arquivo_salvo"] = motor.salvar_resultado(r)
            st.success(f"Rodada concluída e salva no Histórico ({motor.PASTA_HIST}). Veja a aba **Recomendação**.")
        except Exception as e:
            st.warning(f"Rodada concluída, mas não consegui salvar no Histórico: {e}")
    except Exception:
        st.error("Erro ao rodar a simulação:")
        st.code(traceback.format_exc())
    finally:
        barra.empty()

# ---------- Cabeçalho ----------
res = ss["resultado"]
if res is not None:
    st.caption(f"Resultado em tela: **{res['produto']['nome']}** · rodado em {res['data']} · {res['parametros']['n_rep']} "
               f"simulações por cenário · proteção {res['parametros']['protecao']}"
               + (" · aberto do histórico" if res.get("_arquivo") else ""))
else:
    st.caption("Escolha o produto e as análises na barra lateral e clique em **Rodar simulação**.")

# ---------- Cenário em tela (vale para Recomendação, Ação e Análise) ----------
cenario = None
if ss["resultado"] is not None:
    cens = list(ss["resultado"]["cenarios"])
    if ss.get("cenario_ativo") not in cens:
        ss["cenario_ativo"] = cens[0]
    if len(cens) > 1:
        cenario = st.selectbox("Cenário em tela", cens, key="cenario_ativo",
                               help="Combinação de distribuição da demanda, choque e HB que as telas de resultado mostram.")
    else:
        cenario = cens[0]

f = {"produto": produto, "par": par, "resultado": ss["resultado"], "janelas": (up, down), "cenario": cenario}

# ---------- As 6 telas ----------
telas = {"🏠 Início": "tela_inicio", "📝 Cadastro": "tela_cadastro", "✅ Recomendação": "tela_recomendacao",
         "📅 Ação S&OE": "tela_acao", "📊 Análise avançada": "tela_analise", "🗂️ Histórico": "tela_historico"}
abas = st.tabs(list(telas))
importlib.invalidate_caches()  # enxerga arquivos de tela gravados depois que o app começou
for aba, modulo in zip(abas, telas.values()):
    with aba:
        try:
            tela = importlib.import_module(modulo)
        except ModuleNotFoundError as erro:
            if erro.name != modulo:
                raise
            st.info(f"🚧 Tela em construção: grave **{modulo}.py** no notebook e recarregue a página (F5).")
            continue
        try:
            tela.mostrar(dados, f)
        except Exception:
            st.error(f"Erro na tela {modulo}:")
            st.code(traceback.format_exc())

## Passos 7 a 12 — As seis telas
Cada célula grava uma tela. Se uma tela ainda não existir, o app mostra **🚧 em construção** naquela aba.

### Passo 7 —  Início

In [ ]:
%%writefile tela_inicio.py
# tela_inicio.py — Tela 1: o que é a ferramenta, como usar, quem fez e documentos para baixar.
# Os textos ficam em conteudo.py (fácil de editar sem mexer na tela).
import importlib

import streamlit as st

import conteudo
import motor


def mostrar(dados, f):
    importlib.reload(conteudo)  # edições no conteudo.py aparecem ao recarregar a página
    st.header(conteudo.TITULO)
    st.markdown(f"*{conteudo.SUBTITULO}*")

    esq, dir_ = st.columns([3, 2], gap="large")
    with esq:
        st.subheader("O que é")
        st.markdown(conteudo.O_QUE_E)
        st.subheader("Que problema resolve")
        st.markdown(conteudo.PROBLEMA)
        st.subheader("Como usar")
        st.markdown("\n".join(f"{i}. {passo}" for i, passo in enumerate(conteudo.COMO_USAR, 1)))
        st.info(conteudo.JANELA)
    with dir_:
        st.subheader("Quem fez")
        for papel, nomes in conteudo.EQUIPE.items():
            st.markdown(f"**{papel}:** " + " · ".join(nomes))
        st.caption(conteudo.PROJETO)
        st.caption(conteudo.AGRADECIMENTOS)

        st.subheader("Documentos")
        docs = sorted(motor.PASTA_DOCS.glob("*")) if motor.PASTA_DOCS.exists() else []
        docs = [d for d in docs if d.is_file()]
        if not docs:
            st.caption(f"Coloque os PDFs e documentos na pasta **{motor.PASTA_DOCS}** para que apareçam aqui.")
        for d in docs:
            st.download_button(f"⬇ {conteudo.DOCUMENTOS.get(d.name, d.name)}", d.read_bytes(), file_name=d.name,
                               key=f"doc_{d.name}", width="stretch")
        st.download_button("⬇ Planilha modelo para cadastrar produtos", motor.planilha_modelo([motor.produto_novo("Exemplo")]),
                           file_name="SOLARIS_modelo_importacao.xlsx", key="modelo_inicio", width="stretch")

        st.subheader("A base em uso")
        s = dados["series"]
        n_prod = sum(1 for v in s.values() if v["tipo"] == "produto")
        c1, c2, c3 = st.columns(3)
        c1.metric("Produtos", n_prod, border=True)
        c2.metric("Benchmarks", len(s) - n_prod, border=True)
        c3.metric("Cadastrados", len(st.session_state.get("cadastrados", [])), border=True)
        st.caption(f"Erros de previsão de {len(dados['arquivo'])} pares versão × mês dos benchmarks calibram a incerteza.")

    st.divider()
    a, b = st.columns(2, gap="large")
    with a:
        st.subheader("Políticas comparadas")
        for nome, desc in conteudo.POLITICAS.items():
            st.markdown(f"**{nome}.** {desc}")
    with b:
        st.subheader("Como ler as métricas")
        for nome, desc in conteudo.METRICAS.items():
            st.markdown(f"**{nome}.** {desc}")
    st.caption(conteudo.AVISO_DADOS)

### Passo 8 —  Cadastro (produtos da base, criar/editar à mão, subir planilha)
`st.data_editor` deixa digitar a previsão mês a mês; `st.file_uploader` lê a planilha modelo.

In [ ]:
%%writefile tela_cadastro.py
# tela_cadastro.py — Tela 2: ver os produtos da base, criar/editar produto novo (à mão) e subir planilha.
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st

import motor

ss = st.session_state


def _gravar(produtos):
    """Inclui ou atualiza (mesmo nome) produtos na lista de cadastrados e salva no Drive."""
    lista = [p for p in ss["cadastrados"] if p["nome"] not in {q["nome"] for q in produtos}]
    for q in produtos:
        q = dict(q, id=f"novo::{q['nome']}", fonte="novo")
        q.pop("sku_base", None), q.pop("versao", None)
        lista.append(q)
    ss["cadastrados"] = lista
    try:
        motor.salvar_cadastrados(lista)
        return True
    except Exception as e:
        st.warning(f"Produto guardado nesta sessão, mas não consegui salvar no Drive: {e}")
        return False


def _tabela_mensal(p):
    return pd.DataFrame({"Mês": p["meses"], "Previsão de vendas (un.)": p["previsao"],
                         "Produção programada (un.)": p["programada"]})


def ver_base(dados, f):
    series = dados["series"]
    padrao = f["produto"].get("sku_base") or list(series)[0]
    sku = st.selectbox("Série da base", list(series), index=list(series).index(padrao), key="cad_ver_sku")
    s = series[sku]
    c = st.columns(6)
    c[0].metric("Tipo", "Produto" if s["tipo"] == "produto" else "Benchmark", border=True)
    c[1].metric("Origem", s["origem"], border=True)
    c[2].metric("Cobertura-alvo", f"{motor.num(s['cobertura_dias'])} dias", border=True)
    c[3].metric("HB", f"{motor.num(s['hb_dias'])} dias", border=True)
    c[4].metric("Maior LT de insumo", f"{motor.num(s['lt_insumo_semanas'])} sem.", border=True)
    c[5].metric("Lote (inferido)", motor.num(s["lote"]) if s["lote"] else "sem lote", border=True,
                help="Maior quantidade q tal que ≥ 80% das produções planejadas são múltiplos de q (tolerância 2%).")
    st.caption(f"Preço (R$ PF): {motor.num(s['preco'], 2)} · {len(s['versoes'])} versões da previsão "
               f"({s['versoes'][0]} a {s['versoes'][-1]}) · {s['T_real']} meses com realizado.")

    # previsão da primeira versão × realizado
    fig = go.Figure()
    fig.add_bar(x=s["meses"][:s["T_real"]], y=s["realizado"], name="Realizado (sell-in)", marker_color="#8a8f98")
    for r, nome, cor in [(0, "1ª versão da previsão", "#2a78d6"), (min(3, len(s["versoes"]) - 1), "4ª versão", "#eb6834")]:
        fig.add_scatter(x=s["meses"][r:], y=np.nan_to_num(s["V"][r, r:]), name=f"{nome} ({s['versoes'][r]})", mode="lines+markers",
                        line=dict(color=cor))
    fig.update_layout(title=f"{sku}: previsão × realizado", height=340, separators=",.",
                      margin=dict(t=50, b=10, l=10, r=10), legend=dict(orientation="h", y=-0.2))
    st.plotly_chart(fig, width="stretch")

    with st.expander("Triângulos da base (linha = versão do plano; coluna = mês)"):
        for nome, M in [("Previsão de vendas", s["V"]), ("Plano de produção", s["PP"])]:
            st.markdown(f"**{nome}**")
            st.dataframe(pd.DataFrame(M, index=s["versoes"], columns=s["meses"]).style.format("{:,.0f}", na_rep=""), width="stretch")
        st.markdown("**Realizado (última versão)**")
        st.dataframe(pd.DataFrame([s["realizado"], s["producao_real"], s["estoque_real"]], index=["Venda", "Produção", "Estoque"],
                                  columns=s["meses"][:s["T_real"]]).style.format("{:,.0f}", na_rep=""), width="stretch")
    if st.button("Copiar esta série como produto novo (para editar)", key="cad_copiar"):
        ss["_cad_origem"] = f"Copiar de {sku}"
        st.info("Abra **Criar ou editar** para ajustar e salvar a cópia.")


def criar_editar(dados, f):
    series = dados["series"]
    opcoes = ["Produto em branco"] + [f"Copiar de {k}" for k in series] + [f"Editar ✚ {p['nome']}" for p in ss["cadastrados"]]
    if "_cad_origem" in ss:
        ss["cad_origem"] = ss.pop("_cad_origem")
    if ss.get("cad_origem") not in opcoes:
        ss["cad_origem"] = opcoes[0]
    origem = st.selectbox("Começar de", opcoes, key="cad_origem")
    if origem == "Produto em branco":
        a, b = st.columns(2)
        inicio = a.text_input("Primeiro mês (mmm/aaaa)", "jan/2027", key="cad_inicio")
        n = b.number_input("Número de meses", 3, 36, 12, key="cad_n")
        if motor.mes_label(inicio) is None:
            st.error("Mês inválido. Use o formato mmm/aaaa, por exemplo jan/2027.")
            return
        p0 = motor.produto_novo("Novo produto", motor.mes_label(inicio), int(n))
        sig = f"branco_{motor.mes_label(inicio)}_{n}"
    elif origem.startswith("Copiar de "):
        sku = origem[len("Copiar de "):]
        p0 = motor.produto_da_base(series[sku], motor.versoes_validas(series[sku])[0])
        p0["nome"] = f"Cópia de {sku}"
        sig = f"copia_{sku}"
    else:
        nome = origem[len("Editar ✚ "):]
        p0 = next(p for p in ss["cadastrados"] if p["nome"] == nome)
        sig = f"edit_{nome}_{len(p0['meses'])}"

    st.markdown("**Atributos do produto**")
    c = st.columns(4)
    nome = c[0].text_input("Nome do produto", p0["nome"], key=f"nm_{sig}")
    origem_p = c[1].selectbox("Origem da produção", ["Interno", "Terceiro"], index=0 if p0["origem"] == "Interno" else 1, key=f"or_{sig}")
    cob = c[2].number_input("Cobertura-alvo (dias)", 0.0, 365.0, float(p0["cobertura_dias"]), 1.0, key=f"cb_{sig}")
    hb = c[3].number_input("HB (dias)", 0.0, 180.0, float(p0["hb_dias"]), 1.0, key=f"hb_{sig}")
    lt = c[0].number_input("Maior LT de insumo (semanas)", 0.0, 52.0, float(p0["lt_insumo_semanas"]), 1.0, key=f"lt_{sig}")
    preco = c[1].number_input("R$ PF (preço por unidade)", 0.0, 10_000.0, float(p0["preco"]), 0.1, key=f"pr_{sig}")
    lote = c[2].number_input("Lote mínimo (un.; 0 = sem lote)", 0, 10_000_000, int(p0["lote"]), 1, key=f"lo_{sig}")
    est = c[3].number_input("Estoque atual (un.)", 0.0, 1e9, float(p0["estoque_inicial"]), 1.0, key=f"es_{sig}")

    st.markdown("**Previsão de vendas e produção programada por mês** (edite as células; a produção programada é a que "
                "chega no mês)")
    tab = st.data_editor(_tabela_mensal(p0), key=f"ed_{sig}", num_rows="dynamic", width="stretch", hide_index=True,
                         column_config={"Previsão de vendas (un.)": st.column_config.NumberColumn(min_value=0, format="%.0f"),
                                        "Produção programada (un.)": st.column_config.NumberColumn(min_value=0, format="%.0f")})
    tab = tab.dropna(subset=["Mês"])
    meses = [motor.mes_label(m) for m in tab["Mês"]]
    p = dict(p0, nome=nome.strip(), origem=origem_p, cobertura_dias=cob, hb_dias=hb, lt_insumo_semanas=lt, preco=preco,
             lote=int(lote), estoque_inicial=est, meses=meses,
             previsao=[float(x) if pd.notna(x) else 0.0 for x in tab["Previsão de vendas (un.)"]],
             programada=[float(x) if pd.notna(x) else 0.0 for x in tab["Produção programada (un.)"]])
    erros = motor.validar_produto(p)
    if any(m is None for m in meses):
        erros.append("Há meses em formato inválido (use mmm/aaaa).")
    up, down = motor.janelas(p, f["par"])
    st.caption(f"Com estes dados: aumentar exige decidir {up} meses antes; reduzir, {down} meses antes. "
               f"Meses com previsão zero ficam fora das métricas e não acionam gatilho.")
    if p["nome"] in series:
        erros.append("Use um nome diferente dos produtos da base.")
    existe = any(q["nome"] == p["nome"] for q in ss["cadastrados"])
    if st.button("💾 Atualizar produto" if existe else "💾 Salvar produto", type="primary", disabled=bool(erros), key=f"sv_{sig}"):
        _gravar([p])
        ss["_selecionar"] = f"✚ {p['nome']}"
        ss["_cad_origem"] = f"Editar ✚ {p['nome']}"
        st.rerun()
    for e in erros:
        st.caption(f"⚠️ {e}")


def subir_planilha(dados, f):
    st.markdown("Use a **planilha modelo**: aba *Atributos* (uma linha por produto) e aba *Previsoes* (uma linha por produto e mês).")
    st.download_button("⬇ Baixar planilha modelo", motor.planilha_modelo([motor.produto_novo("Exemplo")]),
                       file_name="SOLARIS_modelo_importacao.xlsx", key="modelo_cad")
    arq = st.file_uploader("Planilha preenchida (.xlsx)", type=["xlsx"], key="cad_upload")
    if not arq:
        return
    produtos, avisos = motor.ler_planilha_produtos(arq)
    for a in avisos:
        st.warning(a)
    if not produtos:
        return
    resumo = pd.DataFrame([{"Produto": p["nome"], "Origem": p["origem"], "Cobertura (dias)": p["cobertura_dias"], "HB (dias)": p["hb_dias"],
                            "LT insumo (sem.)": p["lt_insumo_semanas"], "R$ PF": p["preco"], "Lote": p["lote"],
                            "Estoque": p["estoque_inicial"], "Meses": len(p["meses"]),
                            "De": p["meses"][0], "Até": p["meses"][-1], "Previsão total": sum(p["previsao"]),
                            "Problemas": " ".join(motor.validar_produto(p)) or "ok"} for p in produtos])
    st.dataframe(resumo, hide_index=True, width="stretch")
    validos = [p for p in produtos if not motor.validar_produto(p) and p["nome"] not in dados["series"]]
    if st.button(f"Cadastrar {len(validos)} produto(s)", type="primary", disabled=not validos, key="cad_imp"):
        _gravar(validos)
        ss["_selecionar"] = f"✚ {validos[0]['nome']}"
        st.success("Produtos cadastrados. Eles aparecem na barra lateral com o símbolo ✚.")
        st.rerun()


def lista_cadastrados(dados, f):
    lista = ss["cadastrados"]
    if not lista:
        st.info("Nenhum produto cadastrado ainda.")
        return
    st.dataframe(pd.DataFrame([{"Produto": p["nome"], "Origem": p["origem"], "Cobertura (dias)": p["cobertura_dias"],
                                "HB (dias)": p["hb_dias"], "LT insumo (sem.)": p["lt_insumo_semanas"], "R$ PF": p["preco"],
                                "Lote": p["lote"], "Estoque": p["estoque_inicial"], "Meses": f"{p['meses'][0]} a {p['meses'][-1]}",
                                "Previsão total": motor.num(sum(p["previsao"]))} for p in lista]), hide_index=True, width="stretch")
    st.caption(f"Arquivo: {motor.ARQUIVO_PRODUTOS}")
    a, b = st.columns(2)
    a.download_button("⬇ Baixar todos na planilha modelo", motor.planilha_modelo(lista), file_name="SOLARIS_produtos_cadastrados.xlsx",
                      key="cad_baixar")
    apagar = b.multiselect("Apagar produtos", [p["nome"] for p in lista], key="cad_apagar")
    if apagar and b.button(f"🗑 Apagar {len(apagar)} produto(s)", key="cad_apagar_btn"):
        ss["cadastrados"] = [p for p in lista if p["nome"] not in apagar]
        motor.salvar_cadastrados(ss["cadastrados"])
        st.rerun()


def mostrar(dados, f):
    st.subheader("Cadastro de produtos")
    secoes = {"Produtos da base": ver_base, "Criar ou editar": criar_editar, "Subir planilha": subir_planilha,
              "Produtos cadastrados": lista_cadastrados}
    escolha = st.segmented_control("Seção", list(secoes), default="Criar ou editar", key="cad_secao", label_visibility="collapsed")
    secoes[escolha or "Criar ou editar"](dados, f)

### Passo 9 — Recomendação

In [ ]:
%%writefile tela_recomendacao.py
# tela_recomendacao.py — Tela 3: qual política usar, por quê, e se a resposta muda com a sensibilidade.
import pandas as pd
import streamlit as st

import motor

ss = st.session_state


def tabela_politicas(resumo, par):
    t = resumo.copy()
    t["Política"] = t["politica"].map(motor.NOME_POLITICA)
    t["Cumpre metas"] = t.apply(lambda r: "—" if r["politica"] == "PROG" else ("✅" if r["cumpre_metas"] else "❌"), axis=1)
    out = pd.DataFrame({
        "Política": t["Política"], "Cumpre metas": t["Cumpre metas"],
        "Nível de serviço": t["nivel_servico"].map(motor.pct), "Nível garantido (P5)": t["nivel_garantido"].map(motor.pct),
        "% meses com falta": t["meses_falta"].map(lambda v: motor.pct(v, 0)),
        f"% meses cobertura < {par['cob_min']}d": t["meses_cob_baixa"].map(lambda v: motor.pct(v, 0)),
        f"% meses cobertura > {par['cob_max']}d": t["meses_excesso"].map(lambda v: motor.pct(v, 0)),
        "Estoque médio (un.)": t["estoque_medio"].map(motor.num), "Reserva média (un.)": t["reserva_media"].map(motor.num),
        "Estoque final (un.)": t["estoque_final"].map(motor.num), "Unidades reduzidas": t["unidades_cortadas"].map(motor.num),
        "Custo total esperado": t["custo_total"].map(motor.mi), "Custo P95": t["custo_p95"].map(motor.mi)})
    return out


def mostrar(dados, f):
    res = f["resultado"]
    if res is None:
        st.info("Ainda não há resultado. Escolha o produto e as análises na barra lateral e clique em **▶ Rodar simulação** "
                "(ou abra uma rodada salva na aba Histórico).")
        return
    prod, par = res["produto"], res["parametros"]
    if prod["nome"] != f["produto"]["nome"]:
        st.warning(f"Este resultado é de **{prod['nome']}**. O produto escolhido na barra lateral agora é "
                   f"**{f['produto']['nome']}**: clique em Rodar simulação para atualizar.")
    cen = f["cenario"]
    cd = res["cenarios"][cen]
    b = res["melhor"].set_index("cenario").loc[cen]
    resumo = res["resumo"][res["resumo"]["cenario"] == cen]

    # ---------- Cartão principal ----------
    with st.container(border=True):
        st.markdown(f"### Política recomendada: **{motor.NOME_POLITICA[b['como_executar']]}**")
        if b["como_executar"] != b["politica"]:
            st.caption(f"A melhor no Monte Carlo foi {motor.NOME_POLITICA[b['politica']]}; como o produto tem lote mínimo de "
                       f"{motor.num(prod['lote'])} un., a execução usa a versão em lotes.")
        if b["cumpre_metas"]:
            st.success(f"Cumpre as metas (serviço ≥ {motor.pct(par['meta_servico'], 0)} e garantido ≥ "
                       f"{motor.pct(par['meta_garantido'], 0)}) com o **menor custo** entre as que cumprem.")
        else:
            st.error(f"Nenhuma política cumpre as metas neste cenário. A indicada é a de **maior nível garantido**: "
                     f"levar ao S&OP a discussão de meta, cobertura-alvo ou janela.")
        c = st.columns(5)
        c[0].metric("Nível de serviço", motor.pct(b["nivel_servico"]), border=True, help="Média dos futuros simulados.")
        c[1].metric("Nível garantido (P5)", motor.pct(b["nivel_garantido"]), border=True,
                    help="Em 95% dos futuros o nível de serviço fica acima deste valor.")
        c[2].metric("% meses com falta", motor.pct(b["meses_falta"], 0), border=True)
        c[3].metric(f"% meses > {par['cob_max']} dias", motor.pct(b["meses_excesso"], 0), border=True)
        c[4].metric("Custo total esperado", motor.mi(b["custo_total"], 1), border=True,
                    delta=None if pd.isna(b.get("custo_vs_programado")) else f"{motor.pct(b['custo_vs_programado'], 0)} vs programado",
                    delta_color="inverse")
        st.caption(f"Janela de reação neste cenário: para **aumentar**, decidir com **{cd['janela_aumento']} meses** de antecedência; "
                   f"para **reduzir**, com **{cd['janela_reducao']} meses** ({par['regra_janela'].lower()}"
                   f"{', com mês de decisão' if par['mes_decisao'] else ''}). Proteção {par['protecao']} (z = {res['z']}).")

    # ---------- Comparação das políticas ----------
    st.subheader("Comparação das políticas neste cenário")
    st.dataframe(tabela_politicas(resumo, par), hide_index=True, width="stretch")
    st.caption("Métricas de avaliação consideram só os meses com previsão de venda. Custos somam todo o horizonte. "
               "O plano programado segue o plano do sistema até o último mês programado e depois repõe o previsto.")

    # ---------- Sensibilidade ----------
    if len(res["cenarios"]) > 1:
        st.subheader("A recomendação muda com a sensibilidade?")
        m = res["melhor"]
        st.dataframe(pd.DataFrame({
            "Cenário": m["cenario"], "Política": m["como_executar"].map(motor.NOME_POLITICA),
            "Cumpre metas": m["cumpre_metas"].map({True: "✅", False: "❌"}), "Nível de serviço": m["nivel_servico"].map(motor.pct),
            "Garantido (P5)": m["nivel_garantido"].map(motor.pct), "Custo total": m["custo_total"].map(motor.mi)}),
            hide_index=True, width="stretch")
        n_pol = m["como_executar"].nunique()
        st.caption("A mesma política em todos os cenários: recomendação robusta." if n_pol == 1 else
                   f"A política muda entre os cenários ({n_pol} diferentes): a escolha depende da premissa de demanda/choque/HB.")

    # ---------- Segunda opinião ----------
    st.subheader("Segunda opinião: séries parecidas no experimento anterior")
    so = motor.segunda_opiniao(prod, motor.carregar_referencia(), cd["distribuicao"], cd["choque"], cd["hb_dias"])
    if so is None:
        st.caption("Coloque o arquivo solaris_resultados.json na pasta do projeto para ver a segunda opinião.")
    else:
        cols = st.columns(3)
        for col, t in zip(cols, so["top"]):
            col.metric(t["serie"], motor.NOME_POLITICA.get(t["politica"], t["politica"]).split(" · ")[0],
                       f"semelhança {motor.num(t['semelhanca'])}/100", delta_color="off", border=True)
        txt = (f"Confiança **{so['confianca']}** (concordância entre as 3 mais parecidas). Referência: experimento anterior, "
               f"janela simples, distribuição {so['dist_ref']}, HB {so['hb']} dias.")
        if so.get("proprio"):
            txt += f" No experimento anterior, a própria série {prod.get('sku_base')} ficou com **{so['proprio'][0]}**."
        st.caption(txt)
        sim_pol = {t["politica"] for t in so["top"]}
        if b["politica"] not in sim_pol:
            st.info("A simulação deste produto e as séries parecidas apontam políticas diferentes. Vale a simulação "
                    "(é feita com os dados do próprio produto); a segunda opinião serve de alerta para revisar as premissas.")
        with st.expander("Ranking completo de semelhança"):
            r = so["ranking"].rename(columns={"sku": "Série", "kind": "Tipo", "origin": "Origem", "target_days": "Cobertura (dias)",
                                              "max_material_lt_weeks": "LT insumo (sem.)", "forecast_cv": "Variação da previsão (CV)",
                                              "forecast_peak_ratio": "Pico ÷ média", "semelhanca": "Semelhança"})
            st.dataframe(r.round(2), hide_index=True, width="stretch")

    # ---------- Stockpyl ----------
    if res.get("stockpyl") is not None and len(res["stockpyl"]):
        st.subheader("Confirmação no Stockpyl")
        v = res["stockpyl"]
        v = v[v["cenario"] == cen] if "cenario" in v else v
        ok = v["identico"].mean() if len(v) else float("nan")
        c = st.columns(3)
        c[0].metric("Trajetórias idênticas", motor.pct(ok, 0), border=True)
        c[1].metric("Maior diferença de estoque", f"{motor.num(v['maior_diferenca_un'].max(), 2)} un.", border=True)
        c[2].metric("Simulações comparadas", motor.num(len(v)), border=True)
        st.caption("O Stockpyl recebeu as mesmas demandas, choques e decisões de produção do SOLARIS. Trajetórias idênticas "
                   "confirmam que o simulador do SOLARIS reproduz a dinâmica de estoque, pendência e choque do Stockpyl.")
        st.dataframe(v.groupby("politica")[["estoque_medio_solaris", "estoque_medio_stockpyl", "maior_diferenca_un"]].mean()
                     .rename(index=motor.NOME_POLITICA).style.format("{:,.2f}"), width="stretch")

    # ---------- Downloads ----------
    st.subheader("Baixar esta rodada")
    a, b2 = st.columns(2)
    a.download_button("⬇ Excel completo", motor.bytes_excel(res), file_name=f"SOLARIS_{res['id']}.xlsx", key="rec_xlsx",
                      width="stretch")
    b2.download_button("⬇ JSON (para reabrir)", motor.json_resultado(res), file_name=f"SOLARIS_{res['id']}.json", key="rec_json",
                       width="stretch")

### Passo 10 — Ação S&OE
`st.dataframe(df.style.map(...))` pinta os gatilhos; a ata editável é baixada em Excel.

In [ ]:
%%writefile tela_acao.py
# tela_acao.py — Tela 4: o que fazer em cada mês, até quando decidir e quais meses acendem gatilho no S&OE.
import io

import pandas as pd
import plotly.graph_objects as go
import streamlit as st

import motor


def _cor_gatilho(v):
    if v in ("Crítico", "Atenção"):
        return f"background-color: {motor.CORES[v]}; color: white; font-weight: 600"
    return f"color: {motor.CORES['OK']}" if v == "OK" else ""


def _cor_acao(v):
    return {"Produzir mais": "color: #1A5276; font-weight: 600", "Reduzir": "color: #7D3C98; font-weight: 600",
            "Aumento fora do prazo": "color: #C0392B; font-weight: 600", "Congelado": "color: #7F8C8D"}.get(v, "")


def mostrar(dados, f):
    res = f["resultado"]
    if res is None:
        st.info("Rode a simulação (barra lateral) ou abra uma rodada do Histórico para ver o plano de ação.")
        return
    cen = f["cenario"]
    cd = res["cenarios"][cen]
    melhor = res["melhor"].set_index("cenario").loc[cen]
    pols = [p for p in res["resumo"].loc[res["resumo"]["cenario"] == cen, "politica"].unique() if p != "PROG"]
    a, b = st.columns([2, 3])
    pol = a.selectbox("Política do plano", pols, index=pols.index(melhor["como_executar"]) if melhor["como_executar"] in pols else 0,
                      format_func=lambda p: motor.NOME_POLITICA[p] + (" (recomendada)" if p == melhor["como_executar"] else ""),
                      key="acao_pol")
    b.markdown(f"**Janela:** aumentar exige decidir **{cd['janela_aumento']} meses** antes do mês; reduzir, "
               f"**{cd['janela_reducao']} meses** antes. Meses dentro do HB estão congelados.")

    tab = motor.plano_e_acoes(res, cen, pol)
    n_crit, n_at = int((tab["gatilho"] == "Crítico").sum()), int((tab["gatilho"] == "Atenção").sum())
    c = st.columns(4)
    c[0].metric("Meses críticos", n_crit, border=True)
    c[1].metric("Meses em atenção", n_at, border=True)
    c[2].metric("Produção recomendada", motor.num(tab["recomendada"].sum()), border=True,
                delta=f"{motor.num(tab['recomendada'].sum() - tab['programada'].sum())} vs programado", delta_color="off")
    prox = tab[(tab["acao"] != "Manter") & (tab["acao"] != "Congelado") & (tab["previsao"] > 0)]
    c[3].metric("Próxima decisão", prox["mes"].iloc[0] if len(prox) else "—", border=True,
                help="Primeiro mês com previsão em que o recomendado difere do programado.")

    # ---------- Gráfico: programado × recomendado e estoque projetado ----------
    fig = go.Figure()
    fig.add_bar(x=tab["mes"], y=tab["programada"], name="Produção programada", marker_color="#c3c7cc")
    fig.add_bar(x=tab["mes"], y=tab["recomendada"], name=f"Produção recomendada ({pol})", marker_color=motor.COR_POLITICA.get(pol))
    fig.add_scatter(x=tab["mes"], y=tab["previsao"], name="Previsão de vendas", mode="lines+markers", line=dict(color="#52514e", width=2))
    fig.add_scatter(x=tab["mes"], y=tab["estoque_proj_programado"], name="Estoque projetado (programado)", mode="lines",
                    line=dict(color="#8a8f98", dash="dot"))
    fig.add_scatter(x=tab["mes"], y=tab["estoque_proj_recomendado"], name="Estoque projetado (recomendado)", mode="lines",
                    line=dict(color=motor.COR_POLITICA.get(pol), dash="dot"))
    for _, r in tab[tab["gatilho"].isin(["Crítico", "Atenção"])].iterrows():
        fig.add_annotation(x=r["mes"], y=max(r["previsao"], r["recomendada"], r["programada"]), text="●", showarrow=False,
                           font=dict(color=motor.CORES[r["gatilho"]], size=16), yshift=10)
    fig.update_layout(barmode="group", height=380, separators=",.", margin=dict(t=30, b=10, l=10, r=10),
                      legend=dict(orientation="h", y=-0.2), yaxis_title="unidades")
    st.plotly_chart(fig, width="stretch")
    st.caption("Plano recomendado = a política aplicada à previsão atual (como se ela se confirmasse). Os riscos de falta e "
               "de excesso vêm dos futuros simulados com o plano programado.")

    # ---------- Tabela de ações ----------
    so_gat = st.toggle("Mostrar só meses com gatilho ou mudança", value=False, key="acao_filtro")
    t = tab.copy()
    if so_gat:
        t = t[(t["gatilho"].isin(["Crítico", "Atenção"])) | (~t["acao"].isin(["Manter", "Congelado"]))]
    vis = pd.DataFrame({
        "Mês": t["mes"], "Gatilho": t["gatilho"], "Ação": t["acao"], "Previsão": t["previsao"].map(motor.num),
        "Programado": t["programada"].map(motor.num), "Recomendado": t["recomendada"].map(motor.num),
        "Diferença": t["diferenca"].map(motor.num), "Decidir aumento até": t["decidir_aumento_ate"],
        "Decidir redução até": t["decidir_reducao_ate"], "Estoque proj. (programado)": t["estoque_proj_programado"].map(motor.num),
        "Cobertura (programado, dias)": t["cobertura_programado"].map(motor.num),
        "Risco de falta": t["risco_falta"].map(lambda v: motor.pct(v, 0)),
        "Risco de excesso": t["risco_excesso"].map(lambda v: motor.pct(v, 0)), "Motivo": t["motivo"], "O que fazer": t["o_que_fazer"]})
    st.dataframe(vis.style.map(_cor_gatilho, subset=["Gatilho"]).map(_cor_acao, subset=["Ação"]), hide_index=True, width="stretch",
                 height=min(38 * (len(vis) + 1) + 4, 600))
    with st.expander("Regras dos gatilhos"):
        g = motor.GATILHO
        st.markdown(
            f"- **Crítico:** falta projetada com o plano programado, cobertura abaixo de {res['parametros']['cob_min']} dias "
            f"ou risco de falta ≥ {motor.pct(g['risco_critico'], 0)} nos futuros simulados.\n"
            f"- **Atenção:** risco de falta ≥ {motor.pct(g['risco_atencao'], 0)}, cobertura acima de "
            f"{res['parametros']['cob_max']} dias ou risco de excesso ≥ {motor.pct(g['risco_excesso_atencao'], 0)}, ou "
            f"programado diferente do recomendado em {motor.pct(g['desvio_atencao'], 0)} ou mais (fora do HB).\n"
            f"- **Sem gatilho (—):** meses com previsão de vendas igual a zero. Meses com previsão zero dentro do HB nem aparecem.\n"
            f"- **Aumento fora do prazo:** o recomendado é maior que o programado, mas o mês já passou do prazo para aumentar "
            f"(insumo + produção); só é possível reduzir ou buscar alternativas.")

    # ---------- Ata para a reunião ----------
    st.subheader("Ata da reunião de S&OE")
    ata = t[t["gatilho"].isin(["Crítico", "Atenção"]) | (~t["acao"].isin(["Manter", "Congelado"]))][
        ["mes", "gatilho", "acao", "diferenca", "decidir_aumento_ate", "decidir_reducao_ate", "o_que_fazer"]].copy()
    ata = ata.rename(columns={"mes": "Mês", "gatilho": "Gatilho", "acao": "Ação", "diferenca": "Diferença (un.)",
                              "decidir_aumento_ate": "Aumento até", "decidir_reducao_ate": "Redução até", "o_que_fazer": "O que fazer"})
    ata["Responsável"], ata["Status"], ata["Observação"] = "", "Aberto", ""
    ed = st.data_editor(ata, hide_index=True, width="stretch", key=f"ata_{res['id']}_{cen}_{pol}",
                        disabled=["Mês", "Gatilho", "Ação", "Diferença (un.)", "Aumento até", "Redução até", "O que fazer"],
                        column_config={"Status": st.column_config.SelectboxColumn(options=["Aberto", "Em andamento", "Feito", "Não se aplica"]),
                                       "Diferença (un.)": st.column_config.NumberColumn(format="%.0f")})
    st.caption("O editor não salva sozinho: baixe a ata ao fim da reunião.")
    buf = io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as w:
        ed.to_excel(w, sheet_name="Ata", index=False)
        tab.to_excel(w, sheet_name="Plano_completo", index=False)
    a, b = st.columns(2)
    a.download_button("⬇ Ata + plano (Excel)", buf.getvalue(), file_name=f"SOLARIS_acao_{res['id']}_{pol}.xlsx", key="acao_xlsx",
                      width="stretch")
    b.download_button("⬇ Plano (CSV)", tab.to_csv(index=False, sep=";", decimal=",").encode("utf-8-sig"),
                      file_name=f"SOLARIS_plano_{res['id']}_{pol}.csv", key="acao_csv", width="stretch")

### Passo 11 — Análise avançada (gráficos Plotly)

In [ ]:
%%writefile tela_analise.py
# tela_analise.py — Tela 5: gráficos para entender o resultado (faixa de estoque, risco por mês, custo × serviço,
# distribuição do serviço, sensibilidade, curva de incerteza, backtest com o histórico real e confirmação no Stockpyl).
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st

import motor

LAYOUT = dict(height=360, separators=",.", margin=dict(t=50, b=10, l=10, r=10), legend=dict(orientation="h", y=-0.22),
              hovermode="x unified")


def _rgba(hex_, a):
    h = hex_.lstrip("#")
    return f"rgba({int(h[0:2], 16)},{int(h[2:4], 16)},{int(h[4:6], 16)},{a})"


def _nome(p):
    return motor.NOME_POLITICA.get(p, p).split(" · ")[0]


def faixa_estoque(res, cen, pols):
    prod = res["produto"]
    fig = go.Figure()
    fig.add_scatter(x=prod["meses"], y=prod["previsao"], name="Previsão de vendas", mode="lines", line=dict(color="#52514e", width=2, dash="dash"))
    tab = []
    for p in pols:
        m = res["mensal"].get((cen, p))
        if m is None:
            continue
        cor = motor.COR_POLITICA[p]
        fig.add_scatter(x=prod["meses"] + prod["meses"][::-1], y=list(m["estoque_p95"]) + list(m["estoque_p5"])[::-1], fill="toself",
                        fillcolor=_rgba(cor, 0.15), line=dict(width=0), name=f"{_nome(p)} · faixa P5–P95", hoverinfo="skip",
                        showlegend=True)
        fig.add_scatter(x=prod["meses"], y=m["estoque_p50"], name=f"{_nome(p)} · mediana", mode="lines+markers",
                        line=dict(color=cor, width=2), marker=dict(size=8))
        tab.append(m[["estoque_p5", "estoque_p50", "estoque_p95"]].assign(politica=_nome(p), mes=prod["meses"]))
    fig.update_layout(title="Estoque no fim do mês: mediana e faixa de 90% dos futuros", yaxis_title="unidades", **LAYOUT)
    return fig, pd.concat(tab) if tab else pd.DataFrame()


def risco_mensal(res, cen, pols):
    prod = res["produto"]
    fig = go.Figure()
    for p in pols:
        m = res["mensal"].get((cen, p))
        if m is None:
            continue
        cor = motor.COR_POLITICA[p]
        fig.add_scatter(x=prod["meses"], y=m["prob_falta"], name=f"{_nome(p)} · risco de falta", mode="lines+markers",
                        line=dict(color=cor, width=2), marker=dict(size=8))
        fig.add_scatter(x=prod["meses"], y=m["prob_excesso"], name=f"{_nome(p)} · risco de excesso", mode="lines+markers",
                        line=dict(color=cor, width=2, dash="dot"), marker=dict(size=8, symbol="diamond"))
    for y, txt in [(motor.GATILHO["risco_critico"], "crítico"), (motor.GATILHO["risco_atencao"], "atenção")]:
        fig.add_hline(y=y, line_dash="dash", line_color="#8a8f98", line_width=1, annotation_text=txt, annotation_position="top left")
    fig.update_layout(title="Probabilidade de faltar e de passar do limite de excesso, mês a mês", yaxis=dict(tickformat=".0%", range=[0, 1.02]),
                      **LAYOUT)
    return fig


def custo_servico(res, cen_atual):
    r = res["resumo"]
    fig = go.Figure()
    simbolos = ["circle", "square", "diamond", "triangle-up", "x", "star", "hexagon", "cross"]
    cens = list(res["cenarios"])
    for p in r["politica"].unique():
        g = r[r["politica"] == p]
        fig.add_scatter(x=g["custo_total"] / 1e6, y=g["nivel_garantido"], mode="markers", name=_nome(p),
                        marker=dict(color=motor.COR_POLITICA[p], size=[16 if c == cen_atual else 10 for c in g["cenario"]],
                                    symbol=[simbolos[cens.index(c) % len(simbolos)] for c in g["cenario"]],
                                    line=dict(color="white", width=2)),
                        text=g["cenario"], hovertemplate="%{text}<br>custo R$ %{x:,.2f} mi<br>garantido %{y:.1%}<extra>" + _nome(p) + "</extra>")
    meta = res["parametros"]["meta_garantido"]
    fig.add_hline(y=meta, line_dash="dash", line_color="#8a8f98", annotation_text=f"meta garantida {motor.pct(meta, 0)}")
    fig.update_layout(title="Custo total esperado × nível garantido (P5) — todos os cenários", xaxis_title="custo total (R$ mi)",
                      yaxis=dict(tickformat=".0%"), **dict(LAYOUT, hovermode="closest"))
    return fig


def composicao_custo(res, cen):
    r = res["resumo"][res["resumo"]["cenario"] == cen]
    fig = go.Figure()
    partes = [("custo_estoque", "Manter estoque", "#2a78d6"), ("custo_falta", "Falta (pendência)", "#e34948"),
              ("custo_sobra", "Sobra no fim", "#eda100"), ("custo_excesso", "Excesso > 6 meses", "#6250d6")]
    for col, nome, cor in partes:
        fig.add_bar(x=[_nome(p) for p in r["politica"]], y=r[col] / 1e6, name=nome, marker=dict(color=cor, line=dict(color="white", width=2)),
                    hovertemplate="%{x}: R$ %{y:,.2f} mi<extra>" + nome + "</extra>")
    fig.update_layout(barmode="stack", title="De onde vem o custo (média dos futuros)", yaxis_title="R$ mi", **dict(LAYOUT, hovermode="closest"))
    return fig


def distribuicao_servico(res, cen):
    rep = res["repeticoes"]
    rep = rep[rep["cenario"] == cen]
    fig = go.Figure()
    for p in rep["politica"].unique():
        fig.add_box(y=rep.loc[rep["politica"] == p, "nivel_servico"], name=_nome(p), marker_color=motor.COR_POLITICA[p], boxpoints=False)
    fig.add_hline(y=res["parametros"]["meta_servico"], line_dash="dash", line_color="#8a8f98", annotation_text="meta")
    fig.update_layout(title="Nível de serviço em cada futuro simulado", yaxis=dict(tickformat=".0%"), showlegend=False,
                      **dict(LAYOUT, hovermode="closest"))
    return fig


def mapa_melhor(res):
    m = res["melhor"].copy()
    m["linha"] = m["distribuicao"] + " · " + m["choque"]
    m["coluna"] = "HB " + m["hb_dias"].astype(int).astype(str) + " dias"
    z = m.pivot(index="linha", columns="coluna", values="nivel_garantido")
    txt = m.assign(t=m["como_executar"].map(_nome) + "<br>" + m["nivel_garantido"].map(motor.pct) + m["cumpre_metas"].map(
        {True: " ✓", False: " ✗"})).pivot(index="linha", columns="coluna", values="t")
    fig = go.Figure(go.Heatmap(z=z.values, x=list(z.columns), y=list(z.index), text=txt.values, texttemplate="%{text}",
                               colorscale=[[0, "#f3f1ee"], [1, "#2a78d6"]], zmin=0.5, zmax=1, xgap=2, ygap=2,
                               colorbar=dict(title="garantido", tickformat=".0%"),
                               hovertemplate="%{y} · %{x}<br>%{text}<extra></extra>"))
    fig.update_layout(title="Política recomendada e nível garantido em cada cenário", height=120 + 60 * len(z), separators=",.",
                      margin=dict(t=50, b=10, l=10, r=10))
    return fig


def curva(res):
    c = pd.DataFrame(res["curva"])
    fig = go.Figure(go.Bar(x=c["h"], y=c["cv"] * res["parametros"]["mult_incerteza"], marker=dict(color="#2a78d6", cornerradius=4),
                           text=[motor.pct(v, 0) for v in c["cv"] * res["parametros"]["mult_incerteza"]], textposition="outside",
                           hovertemplate="horizonte %{x} meses<br>erro %{y:.0%}<extra></extra>"))
    fig.update_layout(title="Erro da previsão por horizonte (benchmarks)", xaxis_title="meses entre a previsão e o mês-alvo",
                      yaxis=dict(tickformat=".0%", title="desvio do erro ÷ previsão média"), **dict(LAYOUT, hovermode="closest"))
    return fig


def mostrar(dados, f):
    res = f["resultado"]
    if res is None:
        st.info("Rode a simulação (barra lateral) ou abra uma rodada do Histórico para ver os gráficos.")
        return
    cen = f["cenario"]
    disponiveis = [p for p in ["P2", "P3", "P4", "P2L", "PROG"] if (cen, p) in res["mensal"]]
    melhor = res["melhor"].set_index("cenario").loc[cen, "como_executar"]
    pols = st.multiselect("Políticas nos gráficos", disponiveis, default=[p for p in dict.fromkeys([melhor, "PROG"]) if p in disponiveis],
                          format_func=lambda p: motor.NOME_POLITICA[p], key="an_pols")
    if not pols:
        st.warning("Escolha pelo menos uma política.")
        return

    fig, tab = faixa_estoque(res, cen, pols)
    st.plotly_chart(fig, width="stretch")
    with st.expander("Ver dados da faixa de estoque"):
        st.dataframe(tab.round(0), hide_index=True, width="stretch")
    st.plotly_chart(risco_mensal(res, cen, pols), width="stretch")

    a, b = st.columns(2)
    a.plotly_chart(custo_servico(res, cen), width="stretch")
    b.plotly_chart(composicao_custo(res, cen), width="stretch")

    a, b = st.columns(2)
    if res.get("repeticoes") is not None:
        a.plotly_chart(distribuicao_servico(res, cen), width="stretch")
    else:
        a.info("A distribuição por futuro simulado só aparece em rodadas feitas nesta sessão (o histórico guarda os resumos; "
               "as repetições ficam no Excel da rodada).")
    b.plotly_chart(curva(res), width="stretch")

    if len(res["cenarios"]) > 1:
        st.plotly_chart(mapa_melhor(res), width="stretch")

    # ---------- Backtest com o histórico real ----------
    prod = res["produto"]
    serie = dados["series"].get(prod.get("sku_base")) if prod.get("fonte") == "base" else None
    st.subheader("Backtest com o histórico real")
    bt = motor.backtest_historico(serie, res["parametros"], res["z"], pd.DataFrame(res["curva"]), prod.get("versao", 0)) if serie else None
    if bt is None:
        st.caption("Disponível para séries da base com pelo menos 3 meses de venda realizada a partir da versão escolhida: "
                   "aplica cada política às previsões que a empresa de fato tinha em cada mês e à venda real. "
                   "P0 = o que foi produzido de fato.")
    else:
        t = bt["tabela"]
        fig = go.Figure()
        fig.add_bar(x=bt["meses"], y=bt["realizado"], name="Venda realizada", marker=dict(color="#c3c7cc", cornerradius=4))
        for _, r in t.iterrows():
            fig.add_scatter(x=bt["meses"], y=r["_estoque"], name=f"Estoque · {r['politica']}", mode="lines+markers",
                            line=dict(color=motor.COR_POLITICA.get(r["politica"], "#8a8f98"), width=2,
                                      dash="dash" if r["politica"] == "P0" else "solid"), marker=dict(size=8))
        fig.update_layout(title=f"{prod['sku_base']}: estoque de cada política com a demanda real (negativo = pendência)",
                          yaxis_title="unidades", **LAYOUT)
        st.plotly_chart(fig, width="stretch")
        st.dataframe(pd.DataFrame({"Política": t["politica"].map(lambda p: motor.NOME_POLITICA.get(p, p)),
                                   "Nível de serviço": t["nivel_servico"].map(motor.pct), "% meses com falta": t["meses_falta"].map(lambda v: motor.pct(v, 0)),
                                   "% meses > limite": t["meses_excesso"].map(lambda v: motor.pct(v, 0)),
                                   "Estoque médio": t["estoque_medio"].map(motor.num), "Estoque final": t["estoque_final"].map(motor.num),
                                   "Custo total": t["custo_total"].map(motor.mi)}), hide_index=True, width="stretch")
        st.caption("Um único futuro (o que aconteceu): serve para checar se a política faria sentido na prática, não substitui o Monte Carlo.")

    # ---------- Stockpyl ----------
    v = res.get("stockpyl")
    if v is not None and len(v):
        st.subheader("SOLARIS × Stockpyl")
        v = v[v["cenario"] == cen]
        fig = go.Figure()
        for p in v["politica"].unique():
            g = v[v["politica"] == p]
            fig.add_scatter(x=g["estoque_medio_solaris"], y=g["estoque_medio_stockpyl"], mode="markers", name=_nome(p),
                            marker=dict(color=motor.COR_POLITICA[p], size=9, line=dict(color="white", width=2)))
        lim = [0, float(max(v["estoque_medio_solaris"].max(), v["estoque_medio_stockpyl"].max()) * 1.05 or 1)]
        fig.add_scatter(x=lim, y=lim, mode="lines", line=dict(color="#8a8f98", dash="dash", width=1), name="iguais", hoverinfo="skip")
        fig.update_layout(title="Estoque médio por futuro: SOLARIS (x) × Stockpyl (y)", xaxis_title="SOLARIS (un.)",
                          yaxis_title="Stockpyl (un.)", **dict(LAYOUT, hovermode="closest"))
        st.plotly_chart(fig, width="stretch")

### Passo 12 — Histórico

In [ ]:
%%writefile tela_historico.py
# tela_historico.py — Tela 6: todas as rodadas salvas no Drive: consultar, reabrir nas telas, baixar e apagar.
import pandas as pd
import streamlit as st

import motor

ss = st.session_state


def mostrar(dados, f):
    st.subheader("Histórico de rodadas")
    st.caption(f"Pasta: {motor.PASTA_HIST} · cada rodada gera um .json (para reabrir) e um .xlsx (para analisar).")
    h = motor.listar_historico()
    if h.empty:
        st.info("Nenhuma rodada salva ainda. Cada vez que você roda a simulação, o resultado é salvo aqui automaticamente.")
        return

    produtos = sorted(h["produto"].unique())
    filtro = st.multiselect("Filtrar por produto", produtos, key="hist_filtro", placeholder="Todos os produtos")
    if filtro:
        h = h[h["produto"].isin(filtro)]
    st.dataframe(pd.DataFrame({
        "Data": h["data"], "Produto": h["produto"], "Cenário": h["cenario"], "Política": h["como_executar"].map(motor.NOME_POLITICA),
        "Cumpre metas": h["cumpre_metas"].map({True: "✅", False: "❌"}), "Nível de serviço": h["nivel_servico"].map(motor.pct),
        "Garantido (P5)": h["nivel_garantido"].map(motor.pct), "Custo total": h["custo_total"].map(motor.mi),
        "Simulações": h["simulacoes"], "Proteção": h["protecao"], "Stockpyl": h["stockpyl"].map({True: "sim", False: ""}),
        "Arquivo": h["arquivo"]}), hide_index=True, width="stretch")

    rodadas = list(dict.fromkeys(h["arquivo"]))
    rot = {a: f"{h.loc[h['arquivo'] == a, 'data'].iloc[0]} · {h.loc[h['arquivo'] == a, 'produto'].iloc[0]}" for a in rodadas}
    st.markdown("**Consultar uma rodada**")
    nome = st.selectbox("Rodada", rodadas, format_func=rot.get, key="hist_sel", label_visibility="collapsed")
    c = st.columns(4)
    if c[0].button(" Abrir nas telas", type="primary", key="hist_abrir", width="stretch"):
        ss["resultado"] = motor.carregar_resultado(nome)
        st.rerun()
    xlsx, js = motor.bytes_historico(nome, ".xlsx"), motor.bytes_historico(nome, ".json")
    if xlsx:
        c[1].download_button("⬇ Excel", xlsx, file_name=f"{nome}.xlsx", key="hist_xlsx", width="stretch")
    if js:
        c[2].download_button("⬇ JSON", js, file_name=f"{nome}.json", key="hist_json", width="stretch")
    with c[3].popover("🗑 Apagar", width="stretch"):
        st.write(f"Apagar a rodada **{rot[nome]}**? Não dá para desfazer.")
        if st.button("Sim, apagar", key="hist_apagar"):
            motor.apagar_resultado(nome)
            if ss.get("resultado") is not None and ss["resultado"].get("_arquivo") == nome:
                ss["resultado"] = None
            st.rerun()

    st.download_button("⬇ Baixar todo o histórico (.zip)", motor.zip_historico(), file_name="SOLARIS_historico.zip", key="hist_zip")

    st.divider()
    st.markdown("**Reabrir uma rodada a partir de um JSON baixado**")
    arq = st.file_uploader("Arquivo .json de uma rodada", type=["json"], key="hist_up")
    if arq and st.button("Importar para o histórico", key="hist_imp"):
        motor.PASTA_HIST.mkdir(parents=True, exist_ok=True)
        nome_novo = arq.name.rsplit(".", 1)[0].removeprefix("SOLARIS_")
        (motor.PASTA_HIST / f"{nome_novo}.json").write_bytes(arq.getvalue())
        r = motor.carregar_resultado(nome_novo)
        (motor.PASTA_HIST / f"{nome_novo}.xlsx").write_bytes(motor.excel_resultado(r))
        ss["resultado"] = motor.carregar_resultado(nome_novo)
        st.rerun()

## Passo 13 — Publicar com um túnel temporário do `cloudflared`

Os mesmos três movimentos do notebook da disciplina: **(1)** baixar o `cloudflared`, **(2)** ligar o Streamlit em segundo
plano, **(3)** abrir o túnel e pegar o endereço público.

> **Atenção:** qualquer pessoa com o link acessa o app enquanto o túnel estiver ligado, e a base tem dados.
> Compartilhe o link só com a equipe e desligue ao terminar (passo 14). Para restringir por e-mail, preencha `EMAILS_PERMITIDOS`.

### 13.1 Baixar o `cloudflared`

In [ ]:
CLOUDFLARED = Path("cloudflared")
URL_CLOUDFLARED = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

if not CLOUDFLARED.exists():
    urllib.request.urlretrieve(URL_CLOUDFLARED, CLOUDFLARED)
    CLOUDFLARED.chmod(0o755)
print(subprocess.run([f"./{CLOUDFLARED}", "--version"], capture_output=True, text=True).stdout)

### 13.2 Ligar o Streamlit em segundo plano
O processo herda a variável `SOLARIS_PASTA` (passo 1), então o app lê a base e grava o histórico no Drive.

In [ ]:
PORTA = 8501


def streamlit_no_ar(porta=PORTA):
    try:
        return urllib.request.urlopen(f"http://localhost:{porta}/_stcore/health", timeout=2).read() == b"ok"
    except Exception:
        return False


def iniciar_streamlit(arquivo="app.py", porta=PORTA, espera=40):
    global proc_streamlit
    if streamlit_no_ar(porta):
        print("O Streamlit já está no ar.")
        return
    proc_streamlit = subprocess.Popen(
        [sys.executable, "-m", "streamlit", "run", arquivo, "--server.port", str(porta),
         "--server.headless", "true", "--server.runOnSave", "true", "--browser.gatherUsageStats", "false"],
        stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT, env=dict(os.environ, SOLARIS_PASTA=str(PASTA)))
    for _ in range(espera):
        if streamlit_no_ar(porta):
            print(f"Streamlit pronto em http://localhost:{porta} (visível só dentro do Colab).")
            return
        time.sleep(1)
    print("O Streamlit não respondeu. Últimas linhas do log:\n", Path("streamlit.log").read_text()[-1500:])


iniciar_streamlit()

### 13.3 Abrir o túnel e pegar o endereço público
Se a página não abrir de primeira, aguarde 10–20 s e recarregue.

In [ ]:
EMAILS_PERMITIDOS = []  # ex.: ["betina@...", "julia@..."]; vazio = qualquer pessoa com o link


def abrir_tunel(porta=PORTA, espera=45):
    global proc_tunel, URL_PUBLICA
    comando = [f"./{CLOUDFLARED}", "tunnel", "--url", f"http://localhost:{porta}", "--no-autoupdate"]
    for email in EMAILS_PERMITIDOS:
        comando += ["--allowed-mail", email]
    proc_tunel = subprocess.Popen(comando, stdout=open("cloudflared.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(espera):
        time.sleep(1)
        log = Path("cloudflared.log").read_text()
        achado = re.search(r"https://(?!api\.)[a-z0-9-]+\.trycloudflare\.com", log)
        if "failed to request quick Tunnel" in log:
            break
        if achado:
            URL_PUBLICA = achado.group(0)
            display(HTML(f'<p style="font-size:16px">SOLARIS no ar: <a href="{URL_PUBLICA}" target="_blank">{URL_PUBLICA}</a></p>'))
            return URL_PUBLICA
        if proc_tunel.poll() is not None:
            break
    print("O endereço não apareceu. Veja o diagnóstico (13.4).")


abrir_tunel()

### 13.4 Diagnóstico (rode quando algo não funcionar)

| Sintoma | O que fazer |
|---|---|
| "O Streamlit não respondeu" | leia `streamlit.log`: geralmente erro de sintaxe em um `.py` ou base ausente na pasta |
| Endereço não aparece | leia `cloudflared.log`; rode `abrir_tunel()` de novo (cada execução gera um endereço novo) |
| Página do Cloudflare com erro 1033/530 | o túnel ainda não propagou ou o Streamlit caiu: espere, recarregue, confira `streamlit_no_ar()` |
| Uma aba mostra "Erro na tela…" | o erro aparece na própria aba; corrija o arquivo da tela, regrave a célula e recarregue (F5) |
| Mudei o `motor.py` e nada mudou | regrave a célula, rode `parar_tudo()` e depois 13.2 e 13.3 (o motor fica em memória) |
| Colab desconectou | rode de novo os passos 0, 1, 2, as células `%%writefile` e o passo 13 (o histórico continua no Drive) |

In [ ]:
print("Streamlit no ar:", streamlit_no_ar())
print("Túnel ativo:", "proc_tunel" in globals() and proc_tunel.poll() is None)
print("\n--- streamlit.log (fim) ---\n", Path("streamlit.log").read_text()[-1500:] if Path("streamlit.log").exists() else "sem log")
print("\n--- cloudflared.log (fim) ---\n", Path("cloudflared.log").read_text()[-800:] if Path("cloudflared.log").exists() else "sem log")

## Passo 14 — Encerrar, levar para o computador e próximos passos

**Encerrar:** a célula abaixo desliga o túnel e o Streamlit. O link público deixa de funcionar na hora.

**Rodar no seu computador** (mais estável para apresentar): baixe os arquivos do app, coloque na mesma pasta da base e rode

```
pip install streamlit plotly openpyxl stockpyl
streamlit run app.py
```

(sem a variável `SOLARIS_PASTA`, o app usa a própria pasta do código para a base, o histórico e os documentos).

In [ ]:
def parar_tudo():
    for nome in ["proc_tunel", "proc_streamlit"]:
        proc = globals().get(nome)
        if proc is not None and proc.poll() is None:
            proc.terminate()
            proc.wait(timeout=10)
            print(nome, "encerrado.")


# parar_tudo()  # descomente e rode ao terminar

In [ ]:
# Baixar os arquivos do app para o computador (e copiar para o Drive, junto dos dados).
ARQUIVOS_APP = ["motor.py", "conteudo.py", "app.py", "tela_inicio.py", "tela_cadastro.py", "tela_recomendacao.py",
                "tela_acao.py", "tela_analise.py", "tela_historico.py"]
(PASTA / "app").mkdir(exist_ok=True)
for arquivo in ARQUIVOS_APP:
    shutil.copy(arquivo, PASTA / "app" / arquivo)
print("Cópia do código em", PASTA / "app")
try:
    from google.colab import files
    for arquivo in ARQUIVOS_APP:
        files.download(arquivo)
except ImportError:
    print("Fora do Colab: os arquivos já estão na pasta", Path.cwd())

### Checklist antes de apresentar

- [ ] o teste do passo 4 bate com o que o app mostra (mesmo produto, mesmas opções, semente 2026)
- [ ] Stockpyl confirma 100% das trajetórias (marque a opção em *Avançado* e rode)
- [ ] um produto novo cadastrado à mão e outro pela planilha modelo rodam sem erro
- [ ] a rodada aparece no Histórico e o Excel abre
- [ ] PDFs em `SOLARIS/docs` aparecem para download na tela de Início
- [ ] link do túnel testado em outro dispositivo e desligado ao final